# Giai đoạn 10: Hình cho bài báo (bản v2, style trắng)

Vẽ lại **8 hình** trực tiếp từ các file kết quả trên Drive. Không train lại; **CPU là đủ**, mất khoảng 3–5 phút (Hình 1 tải bề mặt não fsaverage5 lần đầu).

| Hình | Nội dung | Dữ liệu đọc từ `runs/` |
|---|---|---|
| 1 | Tổng quan: scalp → model → SEEG lưỡng cực | không cần dữ liệu (não 2D từ nilearn) |
| 2 | Kết quả chính: r từng bệnh nhân, so sánh model, đường calib | `shared_bip_car/readout_combo5_strict_qc_none` (+`_f10`, `_f05`), `shared_no_env/readout_fixed_strict_qc_none`, `readout_combo5`, `readout_fixed`, `nfn_baseline`, LeadFormer |
| 3 | Phân rã lợi ích + bank lọc trước/sau khi học | như trên + `readout_*_arch3_strict_qc_none`, `enc_sub-*.pt` |
| 4 | Độ tin cậy theo contact, coverage, top quarter, longevity | `readout_combo5_strict_qc_none`, `readout_longevity` |
| 5 | Band power không cần SEEG | `zeroshot_power/zeroshot_subjects.json` |
| 6 | Phân tích protocol | `protocol/` |
| 7 | Kiểm định ngoài (ScaleOLDNEW) | `p17/`, `review13/volts_sub-*.json` |
| S1 | Audit kênh ScaleOLDNEW | `chanqc20/chanqc20_channels_v20_1.csv` |

Mỗi hình được lưu trong `MyDrive/iEEG_S2I/paper/figs/` dưới dạng **PDF** (chèn LaTeX), **SVG** (sửa chữ trong Inkscape/Figma), **PNG 600 dpi**, kèm `<tên>_numbers.json` chứa mọi con số in trên hình.

**Thứ tự chạy:** 1 → 2 → 3 (kiểm tra dữ liệu) → 4 (vẽ) → 5 (xem) → 6 (tải về).

In [ ]:
#@title 1. Mount Drive + cài thư viện
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json
ROOT     = '/content/drive/MyDrive/iEEG_S2I'   #@param {type:"string"}
CODE_DIR = f'{ROOT}/code'
RUNS_DIR = f'{ROOT}/runs'
FIG_DIR  = f'{ROOT}/paper/figs'
os.makedirs(f'{CODE_DIR}/s2i', exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.chdir(CODE_DIR)
!apt-get -qq install -y fonts-liberation > /dev/null
!pip -q install nilearn statsmodels
print('OK; code ->', CODE_DIR, '| runs ->', RUNS_DIR, '| figs ->', FIG_DIR)

## 2. Ghi code vẽ hình lên Drive
Bốn file dưới đây thay cho `figures.py` cũ (file cũ vẫn để nguyên, không dùng nữa). Chạy cả 4 ô.

In [ ]:
%%writefile {CODE_DIR}/s2i/figstyle.py
"""
Shared figure style for the paper (Phase 10, v2). Every figure module imports from here.

Design rules (decided 12 Oct 2026, figure_plan.md section 5):
  * final print size: double column 180 mm, single column 88 mm; text 6-8 pt; one sans family
  * no gridlines: 1-2 reference lines with a meaning (zero, baseline mean) instead
  * range frame: spines run from the first to the last tick and sit 3 pt away from the data
  * panel headline = a short conclusion in ink, left-aligned with the axis; details go to the caption
  * colour: warm ink, muted ink-blue for our model, one earth-orange accent per figure, warm greys
  * patients: deterministic beeswarm or paired lines, never random jitter
  * labels are placed next to the data; positions that need hand tuning are read from NUDGE[fig][key]
"""
import glob
import os

import numpy as np
import matplotlib as mpl

mpl.use("Agg")
import matplotlib.pyplot as plt                                    # noqa: E402
import matplotlib.transforms as mtransforms                        # noqa: E402
from matplotlib import font_manager                                # noqa: E402

MM = 1 / 25.4
PT = 1 / 72
W2, W1 = 180 * MM, 88 * MM

# ----------------------------------------------------------------------------- palette
INK, INK2, INK3 = "#1d1d1b", "#55534e", "#8d8a83"   # text / secondary text, ticks / tertiary notes
PATIENT = "#cfcac1"                                  # individual patients (background)
BASE = "#7d7a73"                                     # baselines
FIXED = "#9dbad6"                                    # fixed untrained filter bank
OURS = "#1f4e79"                                     # our model
OURS_SOFT = "#dce6f0"                                # band / fill for our model
ACCENT = "#c4512b"                                   # the one thing to look at (one per figure)
ACCENT_SOFT = "#f6e1d8"
EXT = "#2f7d6d"                                      # external cohort (figure 7)
WASH = "#f3efe8"                                     # warm background band instead of a grid
RULE = "#bdb8ae"                                     # reference lines

LBL_FIXED = "fixed untrained bank"


STYLE = "own"                                        # "own" (rules above) or "sp" (SciencePlots base)


def setup(style="own"):
    global STYLE
    STYLE = style
    for d in ("/usr/share/fonts/truetype/liberation", "/usr/share/fonts/truetype/liberation2"):
        for f in glob.glob(os.path.join(d, "LiberationSans*.ttf")):
            font_manager.fontManager.addfont(f)
    if style == "sp":
        # SciencePlots: boxed axes, inward mirrored ticks, minor ticks, 7 pt sans (nature). No LaTeX needed.
        import scienceplots                                        # noqa: F401  (registers the styles)
        plt.style.use(["science", "no-latex", "nature"])
        mpl.rcParams.update({"savefig.bbox": "standard", "savefig.dpi": 600, "pdf.fonttype": 42,
                             "svg.fonttype": "none", "axes.unicode_minus": True, "axes.titlelocation": "left",
                             "figure.dpi": 150, "text.color": INK})
        return
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "Liberation Sans", "DejaVu Sans"],
        "font.size": 6.5, "axes.labelsize": 6.5, "axes.titlesize": 7, "xtick.labelsize": 6,
        "ytick.labelsize": 6, "legend.fontsize": 6,
        "text.color": INK, "axes.labelcolor": INK, "axes.edgecolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
        "axes.linewidth": 0.5, "xtick.major.width": 0.5, "ytick.major.width": 0.5,
        "xtick.major.size": 2.0, "ytick.major.size": 2.0, "xtick.minor.size": 1.2, "ytick.minor.size": 1.2,
        "xtick.major.pad": 1.8, "ytick.major.pad": 1.8, "axes.labelpad": 2.5, "xtick.direction": "out",
        "ytick.direction": "out", "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
        "axes.titlepad": 5, "axes.titlelocation": "left", "legend.frameon": False,
        "axes.unicode_minus": True, "pdf.fonttype": 42, "svg.fonttype": "none", "savefig.dpi": 600,
        "lines.solid_capstyle": "round", "lines.dash_capstyle": "butt", "figure.dpi": 150,
        "mathtext.fontset": "custom", "mathtext.rm": "sans", "mathtext.it": "sans:italic",
        "mathtext.bf": "sans:bold", "mathtext.sf": "sans",
    })


# ----------------------------------------------------------------------------- axes helpers
def range_frame(ax, x=True, y=True, offset=3):
    """Spines from the first to the last tick, pushed `offset` pt away from the data (no-op in SciencePlots mode)."""
    if STYLE == "sp":
        return
    ax.figure.canvas.draw_idle()
    if y:
        t = [v for v in ax.get_yticks() if ax.get_ylim()[0] - 1e-9 <= v <= ax.get_ylim()[1] + 1e-9]
        if t:
            ax.spines["left"].set_bounds(t[0], t[-1])
        ax.spines["left"].set_position(("outward", offset))
    if x:
        t = [v for v in ax.get_xticks() if ax.get_xlim()[0] - 1e-9 <= v <= ax.get_xlim()[1] + 1e-9]
        if t:
            ax.spines["bottom"].set_bounds(t[0], t[-1])
        ax.spines["bottom"].set_position(("outward", offset))


def bare_y(ax):
    """Category axis on the left: no spine, no ticks."""
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0, pad=3)


def headline(ax, s, tag=None):
    """Short conclusion above the axis, left-aligned with the axis (ink, not grey). `tag` = one line of context
    (level, normalisation...) in small grey type between the headline and the axis."""
    if tag:
        tr = mtransforms.offset_copy(ax.transAxes, fig=ax.figure, x=0, y=2.5, units="points")
        ax.text(0.0, 1.0, tag, transform=tr, fontsize=5.6, color=INK3, va="bottom", ha="left")
    ax.set_title(s, loc="left", color=INK, fontsize=7, pad=11.5 if tag else 5)


def letters(fig, items):
    """Panel letters aligned with the outer edge of each axis' tick labels, top-aligned with its headline.
    items: list of (ax, 'a', dx_mm). dx_mm nudges horizontally (+ right)."""
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    W, H = fig.get_size_inches()
    for ax, s, dx in items:
        bb = ax.get_tightbbox(r).transformed(fig.transFigure.inverted())
        t = ax._left_title if ax._left_title.get_text() else ax.title
        title = t.get_window_extent(r).transformed(fig.transFigure.inverted())
        fig.text(bb.x0 + dx * MM / W, title.y1 + 0.4 * MM / H, s, fontsize=8, fontweight="bold", va="top",
                 ha="left", color=INK)


def ref_line(ax, v, axis="y", **kw):
    st = dict(color=RULE, lw=0.5, ls=(0, (2, 2)), zorder=0)
    st.update(kw)
    (ax.axhline if axis == "y" else ax.axvline)(v, **st)


def beeswarm(v, size_data, max_off=0.42):
    """Deterministic 1-D beeswarm: offsets (in category units) so that points `size_data` apart do not overlap."""
    v = np.asarray(v, float)
    order = np.argsort(v)
    off = np.zeros(len(v))
    placed = []
    step = 0.11
    for i in order:
        for k in range(0, 40):
            o = (k + 1) // 2 * step * (1 if k % 2 else -1) if k else 0.0
            if abs(o) > max_off:
                o = np.sign(o) * max_off
                break
            if all(abs(v[i] - v[j]) >= size_data or abs(o - off[j]) >= step * 0.95 for j in placed):
                break
        off[i] = o
        placed.append(i)
    return off


def boot_ci(v, n=10000, seed=0):
    rng = np.random.default_rng(seed)
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    m = v[rng.integers(0, len(v), (n, len(v)))].mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


def fmt(v, d=3, sign=False):
    s = f"{v:+.{d}f}" if sign else f"{v:.{d}f}"
    return s.replace("-", "\u2212")


def bracket(ax, x, y0, y1, text, color=INK, w=0.06, pad=0.05, fs=6.5, weight="bold", side="right", sub=None):
    """Vertical bracket between y0 and y1 at x (data), with a label beside it."""
    s = 1 if side == "right" else -1
    ax.plot([x, x + s * w, x + s * w, x], [y0, y0, y1, y1], color=color, lw=0.6, solid_joinstyle="miter",
            clip_on=False)
    ym = (y0 + y1) / 2
    ax.text(x + s * (w + pad), ym, text, color=color, fontsize=fs, fontweight=weight, va="center",
            ha="left" if s > 0 else "right", clip_on=False)
    if sub:
        ax.text(x + s * (w + pad), ym, "\n\n" + sub, color=INK2, fontsize=5.8, va="center",
                ha="left" if s > 0 else "right", clip_on=False, linespacing=1.0)


def _sp_box(fig):
    """SciencePlots mode: every visible axis gets its full box back; category axes (major tick length 0) lose the
    automatic minor ticks that would fall between categories."""
    from matplotlib.ticker import NullLocator
    for ax in fig.axes:
        if not ax.axison:
            continue
        for sp in ax.spines.values():
            sp.set_visible(True)
        for axis in (ax.xaxis, ax.yaxis):
            t = axis.get_major_ticks()
            if not t or t[0].tick1line.get_markersize() == 0 or not len(axis.get_majorticklocs()):
                axis.set_minor_locator(NullLocator())


def save(fig, out, name):
    os.makedirs(out, exist_ok=True)
    if STYLE == "sp":
        _sp_box(fig)
    for ext in ("pdf", "svg", "png"):
        fig.savefig(os.path.join(out, f"{name}.{ext}"))
    plt.close(fig)
    print("  ->", os.path.join(out, name + ".{pdf,svg,png}"))

In [ ]:
%%writefile {CODE_DIR}/s2i/render3d.py
"""
Minimal 3D rendering for the paper schematic (Fig 1): a fsaverage5 cortex drawn with matplotlib only.

Orthographic camera, smooth (vertex-normal) Lambert + weak specular shading, painter's sort, one rasterized
PolyCollection; everything else in the figure stays vector. Brain surface: fsaverage5 pial + sulcal depth
(FreeSurfer, via nilearn), cached as .npz after the first call.
"""
import os

import numpy as np
from matplotlib.collections import PolyCollection
from matplotlib.colors import to_rgb


def mix(c0, c1, t):
    """Linear blend of two colours, t in [0, 1] (scalar or array) -> RGB array."""
    c0, c1 = np.array(to_rgb(c0)), np.array(to_rgb(c1))
    t = np.clip(np.asarray(t, float), 0, 1)[..., None]
    return c0 * (1 - t) + c1 * t


def _unit(a):
    a = np.asarray(a, float)
    return a / np.linalg.norm(a)


def camera(az, el):
    """Rows: screen-right, screen-up, toward-viewer, in world (RAS) coordinates. az from +x toward +y, el up."""
    az, el = np.radians(az), np.radians(el)
    f = np.array([np.cos(el) * np.cos(az), np.cos(el) * np.sin(az), np.sin(el)])
    r = np.cross([0.0, 0.0, 1.0], f)
    r /= np.linalg.norm(r)
    return np.stack([r, np.cross(f, r), f])


def face_normals(v, f):
    n = np.cross(v[f[:, 1]] - v[f[:, 0]], v[f[:, 2]] - v[f[:, 0]])
    return n / (np.linalg.norm(n, axis=1, keepdims=True) + 1e-12)


def vertex_normals(v, f):
    fn = np.cross(v[f[:, 1]] - v[f[:, 0]], v[f[:, 2]] - v[f[:, 0]])     # area-weighted
    vn = np.zeros_like(v)
    for k in range(3):
        np.add.at(vn, f[:, k], fn)
    return vn / (np.linalg.norm(vn, axis=1, keepdims=True) + 1e-12)


def subdivide(v, f, vn, vals=None):
    """One midpoint subdivision (4 triangles per face); normals and per-vertex values are interpolated."""
    e = np.sort(np.r_[f[:, [0, 1]], f[:, [1, 2]], f[:, [2, 0]]], axis=1)
    ue, inv = np.unique(e, axis=0, return_inverse=True)
    inv = inv.ravel()
    nv = len(v)
    v2 = np.r_[v, (v[ue[:, 0]] + v[ue[:, 1]]) / 2]
    n2 = np.r_[vn, vn[ue[:, 0]] + vn[ue[:, 1]]]
    n2 /= np.linalg.norm(n2, axis=1, keepdims=True) + 1e-12
    m = nv + inv.reshape(3, -1).T                                         # midpoints of edges 01, 12, 20
    a, b, c = f[:, 0], f[:, 1], f[:, 2]
    f2 = np.r_[np.c_[a, m[:, 0], m[:, 2]], np.c_[m[:, 0], b, m[:, 1]], np.c_[m[:, 2], m[:, 1], c],
               np.c_[m[:, 0], m[:, 1], m[:, 2]]]
    out = [v2, f2, n2]
    if vals is not None:
        out.append(np.r_[vals, (vals[ue[:, 0]] + vals[ue[:, 1]]) / 2])
    return out


LIGHTS = (  # camera-space directions and weights: key (upper left, front), fill (right), top
    (_unit([-0.5, 0.6, 0.62]), 0.78),
    (_unit([0.8, -0.1, 0.5]), 0.22),
    (_unit([0.0, 1.0, 0.2]), 0.12),
)


def mesh_faces(v, f, R, to_mm, base_rgb, vn=None, ambient=0.34, spec=0.10, shininess=24):
    """Shade one opaque mesh (back faces culled). Returns (polys [N,3,2] in mm, rgb [N,3], depth [N])."""
    pc = v @ R.T
    fn = face_normals(v, f) @ R.T
    if vn is not None:
        n = vn[f].mean(1)
        n = (n / (np.linalg.norm(n, axis=1, keepdims=True) + 1e-12)) @ R.T
    else:
        n = fn
    keep = fn[:, 2] > 0.0
    f, n = f[keep], n[keep]
    base = np.asarray(base_rgb, float)
    base = base[keep] if base.ndim == 2 else np.broadcast_to(base, (len(f), 3))
    V = np.array([0, 0, 1.0])
    dif = np.zeros(len(f))
    sp = np.zeros(len(f))
    for L, w in LIGHTS:
        dif += w * np.clip(n @ L, 0, 1)
        sp += w * np.clip(n @ _unit(L + V), 0, 1) ** shininess
    col = np.clip(base * (ambient + dif)[:, None] + spec * sp[:, None], 0, 1)
    return to_mm(pc[:, :2])[f], col, pc[f, 2].mean(1)


def draw_faces(ax, parts, z=2, lw=0.12):
    """Painter's sort over several mesh_faces() outputs; one rasterized PolyCollection."""
    P = np.concatenate([p[0] for p in parts])
    C = np.concatenate([p[1] for p in parts])
    D = np.concatenate([p[2] for p in parts])
    o = np.argsort(D)
    pc = PolyCollection(P[o], facecolors=C[o], edgecolors=C[o], linewidths=lw, zorder=z, antialiaseds=False)
    pc.set_rasterized(True)
    ax.add_collection(pc)
    return pc


def ellipsoid_outline(R, c, radii, s, off, n=360):
    """Screen silhouette of an ellipsoid under the orthographic camera, as an (n, 2) polyline in mm."""
    M = np.diag(radii)                                   # x = c + M u, |u| = 1
    A = R[:2] @ M                                        # screen = A u (+ const)
    U, S, Vt = np.linalg.svd(A)
    t = np.linspace(0, 2 * np.pi, n)
    pts = (U @ np.diag(S) @ np.c_[np.cos(t), np.sin(t)].T).T
    return s * (pts + (R[:2] @ c)) + off


def load_brain(cache=None):
    """fsaverage5 pial surfaces (both hemispheres) with per-vertex sulcal depth, outward-oriented faces."""
    cache = cache or os.path.join(os.path.expanduser("~"), ".cache", "s2i", "fs5_pial.npz")
    if os.path.exists(cache):
        d = np.load(cache)
        return {h: (d[f"v_{h}"], d[f"f_{h}"], d[f"s_{h}"]) for h in ("left", "right")}
    from nilearn import datasets, surface
    fs = datasets.fetch_surf_fsaverage("fsaverage5")
    out, save = {}, {}
    for h in ("left", "right"):
        m = surface.load_surf_mesh(fs[f"pial_{h}"])
        v = np.asarray(m.coordinates if hasattr(m, "coordinates") else m[0], float)
        f = np.asarray(m.faces if hasattr(m, "faces") else m[1], int)
        s = np.asarray(surface.load_surf_data(fs[f"sulc_{h}"]), float)
        if ((v[f].mean(1) - v.mean(0)) * face_normals(v, f)).sum(1).mean() < 0:
            f = f[:, ::-1].copy()
        out[h] = (v, f, s)
        save.update({f"v_{h}": v, f"f_{h}": f, f"s_{h}": s})
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    np.savez_compressed(cache, **save)
    return out


def lateral_2d(n=1100, sigma=1.5, cache=None):
    """Flat lateral view of the right hemisphere (anterior to the right), for 2D vector drawing.

    The hemisphere is rasterized once off-screen (front faces, painter's sort, grey = sulcal depth); returns
    pixel-centre coordinates in brain mm (xs, ys), a smoothed inside-mask M and smoothed sulcal field T (0..1,
    > 0.5 = sulcus). Contour M at 0.5 for the outline and T at 0.5 for the gyral pattern.
    """
    cache = cache or os.path.join(os.path.expanduser("~"), ".cache", "s2i", f"fs5_lat2d_{n}_{sigma:g}.npz")
    if os.path.exists(cache):
        d = np.load(cache)
        return d["xs"], d["ys"], d["M"], d["T"]
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    from matplotlib.figure import Figure
    from scipy.ndimage import gaussian_filter
    v, f, su = load_brain()["right"]
    R = camera(0.0, 0.0)                                  # viewer at +x; screen right = anterior
    vn = vertex_normals(v, f)
    v, f, vn, su = subdivide(v, f, vn, su)
    pc = v @ R.T
    f = f[(face_normals(v, f) @ R.T)[:, 2] > 0]
    P, D = pc[:, :2][f], pc[f, 2].mean(1)
    o = np.argsort(D)
    t = 1 / (1 + np.exp(-su[f].mean(1) / 0.5))
    lo, hi = pc[:, :2].min(0) - 4, pc[:, :2].max(0) + 4
    h = int(round(n * (hi[1] - lo[1]) / (hi[0] - lo[0])))
    fig = Figure(figsize=(n / 100, h / 100), dpi=100)
    FigureCanvasAgg(fig)
    fig.patch.set_facecolor((1.0, 0.0, 0.0))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(lo[0], hi[0])
    ax.set_ylim(lo[1], hi[1])
    ax.axis("off")
    c = np.c_[t, t, t][o]
    ax.add_collection(PolyCollection(P[o], facecolors=c, edgecolors=c, linewidths=0.4, antialiaseds=False))
    fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())[::-1, :, :3].astype(float) / 255
    inside = ((img[:, :, 0] - img[:, :, 1]) < 0.5).astype(float)
    M = gaussian_filter(inside, sigma)
    T = gaussian_filter(img[:, :, 1] * inside, sigma * 1.1) / (gaussian_filter(inside, sigma * 1.1) + 1e-6)
    xs = np.linspace(lo[0], hi[0], img.shape[1])
    ys = np.linspace(lo[1], hi[1], img.shape[0])
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    np.savez_compressed(cache, xs=xs, ys=ys, M=M, T=T)
    return xs, ys, M, T

In [ ]:
%%writefile {CODE_DIR}/s2i/paperfigs.py
"""
Paper figures, version 2 (Phase 10 rework, 12 Oct 2026). Style: s2i/figstyle.py; plan: figure_plan.md.

Each figure reads the saved result files only and writes <name>.{pdf,svg,png} plus <name>_numbers.json, which lists
every number printed on the figure, so that it can be checked against the manuscript.

  fig2_main   a per-patient r: lagged ridge -> fixed untrained bank -> final model (primary level, strict)
              b all models incl. deep baselines (secondary level, session-wise normalisation, same trials)
              c calibration efficiency (primary level, strict; 5 / 10 / 20 % calibration on the same test trials)

Usage
  python -m s2i.paperfigs --runs RUNS_DIR --out FIG_DIR [--only fig2]
"""
import argparse
import glob
import json
import os

import numpy as np

from s2i import figstyle as st
from s2i.figstyle import (ACCENT, BASE, EXT, FIXED, INK, INK2, INK3, MM, OURS, OURS_SOFT, PATIENT, RULE, W2,
                          WASH, plt)

STRICT = "_qc_none"                      # primary level (no additional automatic QC)

# hand-tuned label offsets (data units unless noted); edit after looking at the real-data render
NUDGE = {
    "fig1": {"brain": (34.0, 34.0), "brain_w": 58.0},
    "fig2": {"a_base_lab": (-0.16, 0.0), "a_fixed_lab": (0.0, 0.012), "a_final_lab": (0.13, 0.0),
             "c_note": (0.22, 0.026), "c_dx": 0.07},
}


# ----------------------------------------------------------------------------- loading
def load_dir(path, prefix):
    out = {}
    for f in sorted(glob.glob(os.path.join(path, f"{prefix}sub-*.json"))):
        d = json.load(open(f))
        out[d.get("subject", os.path.basename(f)[len(prefix):-5])] = d
    return out


def load_strict(main_run, name="readout_combo5_strict"):
    """Primary-level strict readouts with every calibration fraction in one `fracs` dict: the 20 % run holds frac
    "0.2"; the 10 % and 5 % runs are separate folders (suffix _f10 / _f05, frac keys "0.1" / "0.05")."""
    D = load_dir(os.path.join(main_run, name + STRICT), "readout_")
    for suf in ("_f10", "_f05"):
        for s, d in load_dir(os.path.join(main_run, name + STRICT + suf), "readout_").items():
            if s in D:
                for f, v in d.get("fracs", {}).items():
                    D[s]["fracs"].setdefault(f, v)
    return D


def model_r(D, model, frac="0.2", key="r"):
    return {s: d["fracs"][frac]["models"][model][key] for s, d in D.items()
            if frac in d.get("fracs", {}) and model in d["fracs"][frac]["models"]}


def paired(*dicts):
    subs = sorted(set.intersection(*[set(d) for d in dicts]))
    return subs, [np.array([d[s] for s in subs], float) for d in dicts]


def find_leadformer(runs):
    for run in sorted(glob.glob(os.path.join(runs, "*"))):
        cfgp = os.path.join(run, "config.json")
        if not os.path.exists(cfgp):
            continue
        cfg = json.load(open(cfgp))
        if cfg.get("target") != "bip" or cfg.get("eeg_ref") != "car" or "calib_frac" not in cfg:
            continue
        out = {}
        for f in glob.glob(os.path.join(run, "fold_sub-*.json")):
            fo = json.load(open(f))
            if "calibrated" in fo:
                out[fo["subject"]] = float(np.nanmean(fo["calibrated"]["r"]))
        if out:
            return out
    return {}


def paired_stats(a, b):
    d = np.asarray(a) - np.asarray(b)
    lo, hi = st.boot_ci(d)
    return {"mean_diff": float(d.mean()), "ci": [lo, hi], "wins": int((d > 0).sum()), "n": int(len(d))}


def dump_numbers(out, name, nums):
    os.makedirs(out, exist_ok=True)
    json.dump(nums, open(os.path.join(out, f"{name}_numbers.json"), "w"), indent=1)


# ----------------------------------------------------------------------------- figure 1 (schematic, no data)
# Every coordinate below is in millimetres on the printed page (the axes fill the figure, 1 data unit = 1 mm).
def _rbox(ax, x, y, w, h, fc, ec="none", lw=0.6, r=1.0, ls="-", z=1):
    from matplotlib.patches import FancyBboxPatch
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={r}", fc=fc, ec=ec, lw=lw,
                                ls=ls, zorder=z))


def _arr(ax, p, q, color=INK2, lw=0.55, ls="-", rad=0.0, head=4.5, z=3):
    from matplotlib.patches import FancyArrowPatch
    ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=head, color=color, lw=lw, linestyle=ls,
                                 connectionstyle=f"arc3,rad={rad}", shrinkA=0, shrinkB=0, zorder=z))


def _sig(n, seed, f=(3, 7, 13, 21)):
    t = np.linspace(0, 1, n)
    g = np.random.default_rng(seed)
    s = sum(g.uniform(0.35, 1) / (1 + 0.08 * fi) * np.sin(2 * np.pi * (fi + g.uniform(-1, 1)) * t + g.uniform(0, 6))
            for fi in f)
    return t, s / np.abs(s).max()


def _wave(ax, xc, yc, w, h, cyc, color, lw=0.7, ls="-", skew=0.0, shift=0.0, z=4):
    t = np.linspace(-1, 1, 200)
    env = np.exp(-((t - shift) * 2.3) ** 2) * (1 + skew * t)
    s = env * np.cos(np.pi * cyc * t + skew * 2.0)
    ax.plot(xc + w / 2 * t, yc + h / 2 * s / np.abs(s).max(), color=color, lw=lw, ls=ls, zorder=z,
            solid_capstyle="round")


def _vdots(ax, x, y, gap=1.0, r=0.22, color=INK3):
    for k in (-1, 0, 1):
        ax.add_patch(plt.Circle((x, y + k * gap), r, fc=color, ec="none", zorder=3))


def _ptitle(ax, x, y, letter, title, tag):
    ax.text(1.6, y, letter, fontsize=8, fontweight="bold", color=INK, va="baseline")
    ax.text(x, y, title, fontsize=7, color=INK, va="baseline")
    ax.text(x, y - 3.3, tag, fontsize=5.6, color=INK3, va="baseline")


def _ucurve(ax, x0, y0, x1, y1, ybot, r=2.2, color=ACCENT, lw=0.8):
    """Arrow from (x0, y0) down to ybot, across, and up into (x1, y1), with rounded corners."""
    from matplotlib.patches import FancyArrowPatch
    from matplotlib.path import Path
    sx = np.sign(x1 - x0)
    v = [(x0, y0), (x0, ybot + r), (x0, ybot), (x0 + sx * r, ybot), (x1 - sx * r, ybot), (x1, ybot), (x1, ybot + r),
         (x1, y1)]
    c = [Path.MOVETO, Path.LINETO, Path.CURVE3, Path.CURVE3, Path.LINETO, Path.CURVE3, Path.CURVE3, Path.LINETO]
    ax.add_patch(FancyArrowPatch(path=Path(v, c), arrowstyle="-|>", mutation_scale=5, color=color, lw=lw, zorder=5))


def fig1_overview(runs, out):
    H = 104.0
    fig = plt.figure(figsize=(W2, H * MM))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 180)
    ax.set_ylim(0, H)
    ax.axis("off")
    TEST = "#d6d1c7"

    # ================= a: deployment-faithful evaluation (one patient's recording, as a row of trials)
    _ptitle(ax, 7, 100.2, "a", "Evaluated the way it would be deployed",
            "one patient's recording as a row of trials · hollow = artefact annotated in the dataset, excluded")
    y0, h, slot, bw, sgap = 84.0, 5.0, 1.62, 1.12, 3.2
    sess = [17, 15, 16, 12]                                   # trials per session (schematic)
    ncal, artef = 12, {21, 34, 35, 49}                        # 20 % of 60 trials; dataset-annotated artefacts
    xs, x, k = [], 7.0, 0
    for si, n in enumerate(sess):
        x_s0 = x
        for _ in range(n):
            xs.append(x)
            x += slot
        ax.text((x_s0 + x - slot + bw) / 2, y0 - 2.6, f"session {si + 1}", fontsize=5.2, color=INK3, ha="center",
                va="top")
        x += sgap
    for i, xi in enumerate(xs):
        if i < ncal:
            ax.add_patch(plt.Rectangle((xi, y0), bw, h, fc=OURS, ec="none", zorder=2))
        elif i == ncal:                                       # one-trial guard gap
            ax.plot([xi + bw / 2] * 2, [y0 - 0.3, y0 - 2.6], color=INK3, lw=0.35)
            ax.text(xi + bw / 2 + 0.6, y0 - 2.6, "1-trial gap", fontsize=5.0, color=INK3, ha="left", va="top")
        elif i in artef:
            ax.add_patch(plt.Rectangle((xi + 0.1, y0 + 0.1), bw - 0.2, h - 0.2, fc="white", ec=RULE, lw=0.4, zorder=2))
        else:
            ax.add_patch(plt.Rectangle((xi, y0), bw, h, fc=TEST, ec="none", zorder=2))
    xc1, xt0, xt1 = xs[ncal - 1] + bw, xs[ncal + 1], xs[-1] + bw
    yb = y0 + h + 1.3
    for (a_, b_, col) in ((xs[0], xc1, OURS), (xt0, xt1, INK2)):
        ax.plot([a_, a_, b_, b_], [yb - 0.7, yb, yb, yb - 0.7], color=col, lw=0.45, solid_joinstyle="miter")
    ax.text(xs[0], yb + 1.1, "calibration: first 20 % of trials (≈ 6 min)", fontsize=5.8, color=OURS, ha="left",
            va="bottom", fontweight="bold")
    ax.text(xt1, yb + 1.1, "test: every later trial, same and later sessions", fontsize=5.8, color=INK2, ha="right",
            va="bottom")
    ax.text(xt1 + 1.4, y0 + h / 2, "time", fontsize=5.2, color=INK3, va="center")
    _arr(ax, (xt1 + 6.6, y0 + h / 2), (xt1 + 9.6, y0 + h / 2), color=INK3, lw=0.45, head=3.5)
    # rules (numbered, hand-wrapped)
    rules = [("bipolar SEEG targets, common-average scalp EEG",), ("normalisation and hyper-parameters from",
             "calibration trials only"), ("no automatic artefact rejection beyond", "the dataset's annotations"),
             ("every contact and every patient reported",), ("statistics over patients (n = 15)",),
             ("two main hypotheses pre-specified and tested", "on an independent cohort (n = 12; Fig. 7)")]
    yy = 99.6
    for i, lines in enumerate(rules):
        col = EXT if i == len(rules) - 1 else INK2
        ax.text(130.0, yy, f"{i + 1}", fontsize=5.4, color=INK3 if i < len(rules) - 1 else EXT, va="baseline")
        for j, s in enumerate(lines):
            ax.text(133.0, yy - j * 2.45, s, fontsize=5.6, color=col, va="baseline")
        yy -= 2.45 * len(lines) + 0.55
    ax.plot([127.6, 127.6], [yy + 1.6, 101.6], color=RULE, lw=0.4)

    # ================= b: model for a new patient
    _ptitle(ax, 7, 73.6, "b", "Temporal filters are shared across patients; the spatial readout is not",
            "the model applied to a new patient")
    zb, zt = 34.4, 67.4
    _rbox(ax, 44.0, zb, 40.0, zt - zb, OURS_SOFT, r=1.6, z=0)
    _rbox(ax, 86.0, zb, 92.0, zt - zb, WASH, r=1.6, z=0)
    ax.text(45.6, zt - 2.3, "shared across patients", fontsize=5.6, color=OURS, fontweight="bold", va="center")
    ax.text(87.6, zt - 2.3, "fitted per patient, on calibration trials only", fontsize=5.6, color=INK2,
            fontweight="bold", va="center")
    # head, top view: scalp electrodes and one SEEG shaft
    cx, cy, R = 18.0, 52.0, 10.8
    th = np.linspace(0, 2 * np.pi, 300)
    ax.fill(cx + R * np.cos(th), cy + R * np.sin(th), color="white", zorder=1)
    ax.plot(cx + R * np.cos(th), cy + R * np.sin(th), color=INK2, lw=0.6, zorder=2)
    ax.plot([cx - 1.6, cx, cx + 1.6], [cy + R - 0.15, cy + R + 1.9, cy + R - 0.15], color=INK2, lw=0.6, zorder=2)
    for s in (-1, 1):
        e = np.linspace(-0.55, 0.55, 40)
        ax.plot(cx + s * (R + 0.9 * np.cos(e * 2.6)), cy + 2.6 * e * 1.4, color=INK2, lw=0.6, zorder=2)
    pos = {"Fp1": (-.30, .86), "Fp2": (.30, .86), "F7": (-.72, .47), "F3": (-.36, .42), "Fz": (0, .40),
           "F4": (.36, .42), "F8": (.72, .47), "T7": (-.88, 0), "C3": (-.44, 0), "Cz": (0, 0), "C4": (.44, 0),
           "T8": (.88, 0), "P7": (-.72, -.47), "P3": (-.36, -.42), "Pz": (0, -.40), "P4": (.36, -.42),
           "P8": (.72, -.47), "O1": (-.30, -.86), "O2": (.30, -.86)}
    E = {k: (cx + .82 * R * u, cy + .82 * R * v) for k, (u, v) in pos.items()}
    for k, (ex, ey) in E.items():
        ax.add_patch(plt.Circle((ex, ey), 0.62, fc=INK if k in ("F4", "C4", "P4") else "white", ec=INK, lw=0.4,
                                zorder=4))
    p0, p1 = np.array([2.6, 49.0]), np.array([16.8, 50.4])          # shaft from outside the head inwards
    ax.plot(*zip(p0, p1), color=INK3, lw=0.9, zorder=3, solid_capstyle="butt")
    for j, tt in enumerate(np.linspace(0.42, 0.97, 6)):
        q = p0 + tt * (p1 - p0)
        ax.add_patch(plt.Rectangle((q[0] - 0.5, q[1] - 0.5), 1.0, 1.0, angle=0, fc=OURS if j >= 4 else INK2,
                                   ec="none", zorder=4))
    ax.text(cx, zb + 4.0, "scalp EEG (6–19 channels)", fontsize=5.4, color=INK2, ha="center", va="center")
    ax.text(cx, zb + 1.6, "SEEG shaft, bipolar pairs", fontsize=5.4, color=OURS, ha="center", va="center")
    # scalp channels -> traces
    ty = {"F4": 59.0, "C4": 53.6, "P4": 44.6}
    for k, yv in ty.items():
        ax.plot([E[k][0] + 0.6, 31.0], [E[k][1], yv], color=INK3, lw=0.3, zorder=3)
        t, s = _sig(160, {"F4": 1, "C4": 2, "P4": 3}[k])
        ax.plot(31.4 + 10.6 * t, yv + 1.5 * s, color=INK, lw=0.5, zorder=3)
    _vdots(ax, 36.7, 49.4)
    # filter bank: a stack of identical cards = the same bank on every channel
    bx, by, bw_, bh = 48.4, 40.8, 22.0, 19.4
    for o in (2.6, 1.3):
        _rbox(ax, bx + o, by + o, bw_, bh, "white", OURS, lw=0.4, r=0.9, z=1)
    _rbox(ax, bx, by, bw_, bh, "white", OURS, lw=0.65, r=0.9, z=2)
    for j, (cyc, sk, sh) in enumerate(((1.4, 0.25, 0.08), (2.8, -0.2, -0.05), (5.0, 0.3, 0.04), (8.5, -0.15, 0.0))):
        yc = by + bh - 3.0 - j * 4.75
        _wave(ax, bx + bw_ / 2, yc, 17.0, 3.4, cyc, FIXED, lw=0.6, ls=(0, (1.6, 1.1)))
        _wave(ax, bx + bw_ / 2, yc, 17.0, 3.4, cyc * (1 + 0.06 * (j - 1.5)), OURS, lw=0.75, skew=sk, shift=sh)
    for k, yv in ty.items():
        _arr(ax, (42.6, yv), (bx - 0.5, by + 3 + (yv - 44.6) / 14.4 * (bh - 6)), color=INK2, lw=0.45, head=3.6)
    ax.text(bx, by - 2.0, "28 filters + 12 envelopes per channel", fontsize=5.1, color=INK2, va="center")
    ax.plot([bx, bx + 3.2], [by - 4.5] * 2, color=FIXED, lw=0.7, ls=(0, (1.6, 1.1)))
    ax.text(bx + 4.0, by - 4.5, "Morlet init", fontsize=5.1, color=INK3, va="center")
    ax.plot([bx + 15.4, bx + 18.6], [by - 4.5] * 2, color=OURS, lw=0.8)
    ax.text(bx + 19.4, by - 4.5, "meta-learned", fontsize=5.1, color=INK3, va="center")
    # features
    for o in range(4):
        ax.add_patch(plt.Rectangle((75.0 + o * 0.9, 46.4 + o * 0.9), 4.6, 8.4, fc="white", ec=OURS, lw=0.45,
                                   zorder=2 + o))
    ax.text(79.0, 44.5, "C × 40", fontsize=5.4, color=OURS, ha="center", va="center")
    _arr(ax, (bx + bw_ + 3.2, 51.5), (74.6, 51.5), color=INK2, lw=0.45, head=3.6)
    # ridge readout (matrix)
    g = np.random.default_rng(7)
    Wm = np.tanh(g.normal(0, 0.9, (9, 6)) + np.outer(np.sin(np.arange(9) / 1.7), np.cos(np.arange(6) / 1.3)))
    mx, my, cs = 90.4, 42.6, 1.95
    for i in range(9):
        for j in range(6):
            v = Wm[i, j]
            ax.add_patch(plt.Rectangle((mx + j * cs, my + i * cs), cs, cs, fc=OURS if v > 0 else BASE,
                                       alpha=0.12 + 0.75 * abs(v), ec="white", lw=0.35, zorder=2))
    ax.text(mx, my - 2.0, "ridge readout W", fontsize=5.4, color=INK, va="center")
    ax.text(mx, my - 3.4, "penalty by\nblocked 5-fold r", fontsize=5.1, color=INK3, va="top", linespacing=1.15)
    _arr(ax, (82.4, 51.5), (mx - 0.6, 51.5), color=INK2, lw=0.45, head=3.6)
    # outputs: recorded vs predicted bipolar SEEG
    ox0, ox1 = 107.6, 131.0
    for j, yv in enumerate((58.4, 52.9, 44.6)):
        t, s = _sig(220, 20 + j, f=(2, 5, 9, 16))
        p = np.convolve(s, np.ones(9) / 9, mode="same") * 0.78 + 0.1 * _sig(220, 40 + j, f=(11, 23))[1]
        ax.plot(ox0 + (ox1 - ox0) * t, yv + 1.7 * s, color=TEST, lw=1.25, zorder=2, solid_capstyle="round")
        ax.plot(ox0 + (ox1 - ox0) * t, yv + 1.7 * p, color=OURS, lw=0.6, zorder=3)
    _vdots(ax, (ox0 + ox1) / 2, 49.0)
    ax.plot([ox0, ox0 + 3.4], [62.4] * 2, color=TEST, lw=1.25)
    ax.text(ox0 + 4.1, 62.4, "recorded", fontsize=5.1, color=INK3, va="center")
    ax.plot([ox0 + 13.2, ox0 + 16.6], [62.4] * 2, color=OURS, lw=0.7)
    ax.text(ox0 + 17.3, 62.4, "predicted", fontsize=5.1, color=INK3, va="center")
    _arr(ax, (mx + 6 * cs + 1.0, 51.5), (ox0 - 1.2, 51.5), color=INK2, lw=0.45, head=3.6)
    ax.text(ox0 - 1.0, my - 2.0, "bipolar SEEG, every contact", fontsize=5.4, color=INK, va="center")
    ax.text(ox0 - 1.0, my - 4.5, "final: 5 encoders averaged, mixed with ridge", fontsize=5.1, color=INK3,
            va="center")
    # contact ranking from calibration cross-validation
    rx = 147.0
    vals = np.array([.43, .39, .36, .31, .28, .26, .22, .19, .16, .12, .09, .05])
    for i, v in enumerate(vals):
        yv = 58.8 - i * 1.42
        ax.add_patch(plt.Rectangle((rx, yv - 0.48), v / vals[0] * 25.0, 0.96, fc=OURS if i < 3 else TEST, ec="none",
                                   zorder=2))
    ax.plot([rx - 0.9, rx - 1.6, rx - 1.6, rx - 0.9], [58.8 + 0.6, 58.8 + 0.6, 58.8 - 2 * 1.42 - 0.6,
            58.8 - 2 * 1.42 - 0.6], color=OURS, lw=0.45, solid_joinstyle="miter")
    ax.text(rx - 2.4, 58.8 - 1.42, "top\nquarter", fontsize=5.0, color=OURS, ha="right", va="center",
            linespacing=1.05)
    ax.text(rx, 62.4, "contacts ranked by calibration r", fontsize=5.1, color=INK3, va="center")
    ax.text(rx, my - 2.0, "which contacts to trust", fontsize=5.4, color=INK, va="center")
    ax.text(rx, my - 4.5, "before any test data", fontsize=5.1, color=INK3, va="center")
    _arr(ax, (ox1 + 1.2, 51.5), (rx - 7.2, 51.5), color=INK3, lw=0.45, ls=(0, (2, 1.5)), head=3.6)

    # ================= c: episodic meta-training
    _ptitle(ax, 7, 30.8, "c", "The filters are meta-learned through the ridge readout",
            "episodic training on the other patients (leave-one-patient-out)")
    # patient pool
    gx, gy = [7.0, 10.0, 13.0, 16.0, 19.0], [21.6, 18.1, 14.6]
    role = {(1, 2): "test", (2, 0): "stop", (0, 3): "stop"}
    for r_, yv in enumerate(gy):
        for c_, xv in enumerate(gx):
            k = role.get((r_, c_), "train")
            if k == "test":
                ax.add_patch(plt.Circle((xv, yv), 1.05, fc="white", ec=INK, lw=0.7, zorder=3))
            elif k == "stop":
                ax.add_patch(plt.Circle((xv, yv), 1.0, fc="white", ec=BASE, lw=0.45, ls=(0, (1.2, 0.8)), zorder=3))
            else:
                ax.add_patch(plt.Circle((xv, yv), 1.0, fc=BASE, ec="none", zorder=3))
    for j, (k, s) in enumerate((("test", "held-out test patient"), ("stop", "2 for early stopping"),
                                ("train", "12 for training"))):
        yv = 10.0 - j * 2.7
        if k == "test":
            ax.add_patch(plt.Circle((7.0, yv), 0.8, fc="white", ec=INK, lw=0.6))
        elif k == "stop":
            ax.add_patch(plt.Circle((7.0, yv), 0.75, fc="white", ec=BASE, lw=0.45, ls=(0, (1.2, 0.8))))
        else:
            ax.add_patch(plt.Circle((7.0, yv), 0.75, fc=BASE, ec="none"))
        ax.text(8.8, yv, s, fontsize=5.2, color=INK2, va="center")
    _arr(ax, (21.4, 18.1), (28.6, 18.1), color=INK2, lw=0.45, head=3.6)
    ax.text(25.0, 19.3, "sample 2", fontsize=5.0, color=INK3, ha="center", va="bottom")
    # episode: support / gap / query
    ex0, es, ebw, ey, eh = 30.0, 1.5, 1.05, 15.6, 5.0
    for i in range(20):
        xi = ex0 + i * es
        if i < 11:
            ax.add_patch(plt.Rectangle((xi, ey), ebw, eh, fc=OURS_SOFT, ec=OURS, lw=0.35))
        elif i > 11:
            ax.add_patch(plt.Rectangle((xi, ey), ebw, eh, fc="white", ec=OURS, lw=0.35))
    ax.text(ex0, ey + eh + 1.0, "support\n≈ 4 min", fontsize=5.2, color=OURS, va="bottom", linespacing=1.1)
    ax.text(ex0 + 20 * es - (es - ebw), ey + eh + 1.0, "query\n8 trials", fontsize=5.2, color=OURS, va="bottom",
            ha="right", linespacing=1.1)
    ax.text(ex0 + 11 * es + ebw / 2, ey - 1.4, "gap", fontsize=4.8, color=INK3, ha="center", va="top")
    # encoder chip -> ridge -> loss
    cx0, cw, cy0, ch = 66.0, 13.0, 13.6, 9.0
    _rbox(ax, cx0, cy0, cw, ch, "white", OURS, lw=0.65, r=0.9, z=2)
    for j, cyc in enumerate((1.6, 3.4, 7.0)):
        _wave(ax, cx0 + cw / 2, cy0 + ch - 1.8 - j * 2.7, 10.0, 1.9, cyc, OURS, lw=0.6, skew=0.15 * (j - 1))
    ax.text(cx0 + cw / 2, cy0 + ch + 1.2, "filter bank", fontsize=5.2, color=OURS, ha="center", va="bottom")
    _arr(ax, (ex0 + 20 * es + 0.6, 18.1), (cx0 - 0.5, 18.1), color=INK2, lw=0.45, head=3.6)
    rx0, rw = 85.0, 30.0
    _rbox(ax, rx0, cy0, rw, ch, "white", BASE, lw=0.55, r=0.9, z=2)
    ax.text(rx0 + rw / 2, cy0 + ch - 2.5, "closed-form ridge on support", fontsize=5.6, color=INK, ha="center",
            va="center")
    ax.text(rx0 + rw / 2, cy0 + 2.8, r"$W = (X^{\mathrm{T}}X + \lambda I)^{-1}X^{\mathrm{T}}Y$", fontsize=5.6, color=INK2,
            ha="center", va="center")
    _arr(ax, (cx0 + cw + 0.6, 18.1), (rx0 - 0.5, 18.1), color=INK2, lw=0.45, head=3.6)
    lx0, lw_ = 121.0, 22.0
    _rbox(ax, lx0, cy0, lw_, ch, "white", BASE, lw=0.55, r=0.9, z=2)
    ax.text(lx0 + lw_ / 2, cy0 + ch - 2.5, "predict query", fontsize=5.6, color=INK, ha="center", va="center")
    ax.text(lx0 + lw_ / 2, cy0 + 2.8, "loss = 1 − mean r", fontsize=5.6, color=INK2, ha="center", va="center")
    _arr(ax, (rx0 + rw + 0.6, 18.1), (lx0 - 0.5, 18.1), color=INK2, lw=0.45, head=3.6)
    # the accent: gradient back through the closed-form solution into the filters (and lambda)
    _ucurve(ax, lx0 + lw_ / 2, cy0 - 0.3, cx0 + cw / 2, cy0 - 0.4, 6.4)
    _arr(ax, (rx0 + rw / 2 + 6.5, 6.4), (rx0 + rw / 2 + 6.5, cy0 - 0.4), color=ACCENT, lw=0.6, head=4.2, z=5)
    ax.text(rx0 + rw / 2 + 7.6, 9.6, "λ", fontsize=5.4, color=ACCENT, va="center")
    ax.text((cx0 + lx0 + lw_) / 2, 4.6, "gradient back-propagated through the ridge solution", fontsize=5.6,
            color=ACCENT, ha="center", va="top")
    # take-home
    for j, s in enumerate(("The readout is refitted in every", "episode, so only what transfers",
                           "across patients is learned:", "the temporal filters.")):
        ax.text(148.0, 22.2 - j * 2.6, s, fontsize=5.6, color=INK if j == 3 else INK2, va="baseline",
                fontweight="bold" if j == 3 else "normal")
    ax.plot([146.2, 146.2], [22.2 - 3 * 2.6 - 0.8, 22.2 + 2.0], color=RULE, lw=0.4)

    st.save(fig, out, "fig1_overview")
    dump_numbers(out, "fig1_overview", {"calibration_label_min": "≈ 6 (manuscript: about 6 min; primary 5.7)",
                                         "support_s": 240, "query_trials": 8, "train_patients": 12,
                                         "early_stop_patients": 2, "features_per_channel": 40,
                                         "external_n": 12, "n_patients": 15})


# ----------------------------------------------------------------------------- figure 1, v3 (white scene)
# One left-to-right scene, few words; detail goes to the caption. 1 data unit = 1 mm on the page.
def _trace(n, seed, k=30, alpha=1.0):
    g = np.random.default_rng(seed)
    t = np.linspace(0, 1, n)
    y = sum(np.sin(2 * np.pi * f * 1.6 * t + g.uniform(0, 6.3)) / f ** (0.5 * alpha) for f in range(1, k))
    return y / np.abs(y).max()


FIG1_CH = ("#1a9a8b", "#7051a6", "#3f88c5", "#d39a27")   # four scalp channels (teal, violet, azure, ochre)
BRAIN_GY, BRAIN_SU, BRAIN_LN = "#f7ebe5", "#ead2c8", "#d6b2a5"
PATIENT_DK = "#bdb6aa"                               # recorded SEEG trace (a touch darker than PATIENT)


def _wt_cmap():
    from matplotlib.colors import LinearSegmentedColormap
    return LinearSegmentedColormap.from_list("wt", ["#d39a27", "#fbf9f5", OURS])


def _seq_cmap():
    from matplotlib.colors import LinearSegmentedColormap
    return LinearSegmentedColormap.from_list("q", ["#eef4fa", "#6aa6d6", OURS])


def _brain2d(ax, cx, cy, w, z=2.0, outline_only=False, fc=None, ec=INK2, lw=0.6, ls="-"):
    """Flat lateral right hemisphere (fsaverage5, anterior right) centred at (cx, cy), w mm wide.
    Returns (outline polyline in mm, bbox (x0, y0, x1, y1))."""
    import contourpy
    from matplotlib.patches import Polygon
    from s2i import render3d as r3
    xs, ys, M, T = r3.lateral_2d()
    raw = max(contourpy.contour_generator(xs, ys, M).lines(0.5), key=len)
    x0, x1, y0, y1 = raw[:, 0].min(), raw[:, 0].max(), raw[:, 1].min(), raw[:, 1].max()
    s = w / (x1 - x0)
    X, Y = cx + s * (xs - (x0 + x1) / 2), cy + s * (ys - (y0 + y1) / 2)
    out = np.c_[cx + s * (raw[:, 0] - (x0 + x1) / 2), cy + s * (raw[:, 1] - (y0 + y1) / 2)]
    bb = (out[:, 0].min(), out[:, 1].min(), out[:, 0].max(), out[:, 1].max())
    if outline_only:
        ax.add_patch(Polygon(out, closed=True, fc=fc or "none", ec=ec, lw=lw, ls=ls, zorder=z))
        return out, bb
    poly = Polygon(out, closed=True, fc=BRAIN_GY, ec="none", zorder=z)
    ax.add_patch(poly)
    cs = ax.contourf(X, Y, np.where(M > 0.2, T, 0.0), levels=[0.45, 2.0], colors=[BRAIN_SU], zorder=z + 0.1)
    cs.set_clip_path(poly)
    cl = ax.contour(X, Y, np.where(M > 0.2, T, 0.0), levels=[0.45], colors=[BRAIN_LN], linewidths=0.3,
                    zorder=z + 0.2)
    cl.set_clip_path(poly)
    ax.plot(out[:, 0], out[:, 1], color=ec, lw=lw, zorder=z + 0.3)
    return out, bb


def _head2d(ax, cx, cy, w, q, pad=(2.5, 7.0)):
    """Flat head: brain, scalp arc with electrodes (4 coloured = shown channels) and one SEEG shaft whose
    contacts carry the same colours as the SEEG panel. Sizes scale with w."""
    out, (x0, y0, x1, y1) = _brain2d(ax, cx, cy, w, lw=0.7)
    bw_, bh_ = x1 - x0, y1 - y0
    k = w / 40.0
    hc = np.array([cx, cy + 0.10 * bh_])
    hr = np.array([bw_ / 2 + pad[0], bh_ / 2 + pad[1]])
    t = np.radians(np.linspace(-22, 202, 300))
    ax.plot(hc[0] + hr[0] * np.cos(t), hc[1] + hr[1] * np.sin(t), color=INK3, lw=0.6, zorder=3)
    on = lambda a: hc + hr * np.array([np.cos(np.radians(a)), np.sin(np.radians(a))])      # noqa: E731
    for a in (178, 160, 142, 124, 106):
        ax.add_patch(plt.Circle(on(a), 0.55 * k, fc="white", ec=INK3, lw=0.45, zorder=4))
    for a, c in zip((84, 62, 40, 18), FIG1_CH):
        ax.add_patch(plt.Circle(on(a), 0.85 * k, fc=c, ec="white", lw=0.45, zorder=4.2))
    # SEEG shaft: in through the vertex, tip in the temporal lobe
    pe = on(114) + np.array([0.0, 1.8 * k])
    pt = np.array([x0 + 0.60 * bw_, y0 + 0.24 * bh_])
    ax.plot([pe[0], pt[0]], [pe[1], pt[1]], color=INK, lw=0.85, zorder=5, solid_capstyle="butt")
    u = (pe - pt) / np.linalg.norm(pe - pt)
    cm = _seq_cmap()
    for j, qk in enumerate(q):                            # j = 0 shallowest ... last = tip
        p = pt + u * 1.45 * k * (len(q) - 1 - j)
        ax.add_patch(plt.Circle(p, 0.55 * k, fc=cm(qk), ec=INK, lw=0.35, zorder=5.1))


def fig1_scene(runs, out):
    """Two-tier scene: the patient's head spans the full height on the left; the model runs along the top
    (scalp EEG -> shared filters -> readout) and drops into the predicted SEEG along the bottom."""
    H = 74.0
    yh = 70.0                                             # baseline of the top-row labels
    fig = plt.figure(figsize=(W2, H * MM))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 180)
    ax.set_ylim(0, H)
    ax.axis("off")
    N = NUDGE["fig1"]
    fs, fs2 = 6.5, 5.6
    q = np.array([0.82, 0.35, 0.58, 0.95, 0.70, 0.22, 0.45, 0.30])     # schematic per-contact accuracy
    mid = 57.25                                           # top-row flow line

    # ---- new patient: big flat head on the left, full height
    _head2d(ax, *N["brain"], w=N["brain_w"], q=q)
    ax.text(N["brain"][0], 3.6, "new patient", ha="center", fontsize=fs, color=INK)

    # ---- scalp EEG: four channels in the electrode colours, gathered by a bracket into the filters
    tx0, tx1 = 69.5, 94.0
    ys = [mid + 8.25, mid + 2.75, mid - 2.75, mid - 8.25]
    for j, (y, c) in enumerate(zip(ys, FIG1_CH)):
        x = np.linspace(tx0, tx1, 200)
        ax.plot(x, y + 1.6 * _trace(200, 10 + j), color=c, lw=0.65, zorder=3)
    xb_ = tx1 + 1.6
    ax.plot([xb_ - 0.8, xb_, xb_, xb_ - 0.8], [ys[0] + 1.9, ys[0] + 1.9, ys[-1] - 1.9, ys[-1] - 1.9],
            color=INK3, lw=0.5, zorder=3, solid_joinstyle="miter")
    ax.text((tx0 + tx1) / 2, yh, "scalp EEG", ha="center", fontsize=fs, color=INK)

    # ---- the shared, meta-learned filter bank (the idea of the paper) + the 14 patients behind it
    bx, bw, bh = 100.0, 34.0, 19.0
    by = mid - bh / 2
    _arr(ax, (xb_, mid), (bx - 0.6, mid), color=INK2, lw=0.55, head=4)
    for d in (1.8, 0.9):
        _rbox(ax, bx + d, by + d, bw, bh, "white", ec=OURS, lw=0.4, r=1.2, z=2)
    _rbox(ax, bx, by, bw, bh, OURS_SOFT, ec=OURS, lw=0.75, r=1.2, z=3)
    for j, (cyc, sk) in enumerate(((2.2, 0.0), (3.6, 0.25), (5.6, -0.2))):
        yc = mid + 5.6 - 5.6 * j
        _wave(ax, bx + bw / 2, yc, bw - 8, 4.4, cyc, FIXED, lw=0.6, ls=(0, (1.6, 1.1)), z=4)
        _wave(ax, bx + bw / 2, yc, bw - 8, 4.4, cyc * 1.12, OURS, lw=1.0, skew=sk, shift=0.06 * (j - 1), z=5)
    ax.text(bx + bw / 2, yh, "shared temporal filters", ha="center", fontsize=fs, color=OURS, fontweight="bold")
    for i in range(14):                                   # 12 training (filled), 2 early stopping (dashed)
        es = i in (5, 10)
        _brain2d(ax, bx + 1.25 + i * (bw - 2.5) / 13, by - 2.7, 2.25, z=3, outline_only=True,
                 fc="white" if es else "#a9c1da", ec=INK3 if es else OURS, lw=0.3, ls=(0, (1.0, 0.7)) if es else "-")
    ax.text(bx + bw / 2 + 1.5, by - 6.6, "learned on 14 other patients", ha="center", fontsize=fs2, color=INK2)

    # ---- per-patient readout (fitted on the first 20 % of trials)
    rx, rw, rh = 146.0, 16.0, 22.0
    ry = mid - rh / 2
    g = np.random.default_rng(4)
    Wm = np.tanh(g.normal(0, 1, (9, 6)) * 0.9)
    ax.imshow(Wm, extent=[rx, rx + rw, ry, ry + rh], cmap=_wt_cmap(), vmin=-1, vmax=1, zorder=3,
              interpolation="nearest", aspect="auto")
    ax.add_patch(plt.Rectangle((rx, ry), rw, rh, fc="none", ec=INK2, lw=0.5, zorder=4))
    ax.text(rx + rw / 2 + 2.0, yh, "per-patient readout", ha="center", fontsize=fs, color=INK)
    ax.text(rx + rw + 1.6, mid, "first\n20 % of\ntrials", ha="left", va="center", fontsize=fs2, color=OURS,
            linespacing=1.1)

    # ---- forward pass (grey) and, just under it, the one accent: the gradient back through the ridge
    x_a, x_b = bx + bw + 3.0, rx - 1.0
    yg = mid - 3.8
    _arr(ax, (x_a, mid), (x_b, mid), color=INK2, lw=0.55, head=4)
    _arr(ax, (x_b, yg), (x_a, yg), color=ACCENT, lw=1.05, head=5.5, z=6)
    ax.text((x_a + x_b) / 2, yg - 1.5, "gradient\nthrough\nridge", ha="center", va="top", fontsize=fs2,
            color=ACCENT, linespacing=1.05)

    # ---- bipolar SEEG along the bottom, contacts coloured by how well they are predicted (as in the brain)
    shx, y0, y1 = 73.0, 3.5, 37.5
    cy_ = np.linspace(y1 - 2.0, y0 + 2.0, len(q))
    cm = _seq_cmap()
    ax.plot([shx, shx], [y0, y1], color=INK, lw=0.85, zorder=3, solid_capstyle="butt")
    for j, (yc, qk) in enumerate(zip(cy_, q)):
        ax.add_patch(plt.Rectangle((shx - 1.25, yc - 1.05), 2.5, 2.1, fc=cm(qk), ec=INK, lw=0.4, zorder=4))
        xx = np.linspace(shx + 3.2, 178.5, 320)
        rec = _trace(320, 40 + j, k=40, alpha=1.3)
        pred = qk * rec + (1 - qk) * _trace(320, 80 + j, k=40, alpha=1.3)
        pred = pred / np.abs(pred).max()
        ax.plot(xx, yc + 1.5 * rec, color=PATIENT_DK, lw=1.0, zorder=3, solid_capstyle="round")
        ax.plot(xx, yc + 1.5 * pred, color=OURS, lw=0.6, alpha=0.35 + 0.65 * qk, zorder=4)
    _arr(ax, (rx + rw / 2, ry - 0.8), (rx + rw / 2, y1 + 1.2), color=INK2, lw=0.55, head=4)
    ax.text(shx - 3.2, (y0 + y1) / 2, "bipolar SEEG", ha="center", va="center", rotation=90, fontsize=fs,
            color=INK)
    ly = 41.0
    ax.plot([shx - 3.0, shx + 0.0], [ly] * 2, color=OURS, lw=0.6)
    ax.text(shx + 0.6, ly, "predicted", fontsize=fs2, color=INK2, va="center")
    ax.plot([shx + 12.2, shx + 15.2], [ly] * 2, color=PATIENT_DK, lw=1.0)
    ax.text(shx + 15.8, ly, "recorded", fontsize=fs2, color=INK2, va="center")

    st.save(fig, out, "fig1_overview")
    dump_numbers(out, "fig1_overview", {"schematic": True, "calibration_fraction": 0.2, "other_patients": 14,
                                         "early_stop_patients": 2})


def st_mix(c0, c1, t):
    from s2i.render3d import mix
    return tuple(mix(c0, c1, t))


def _div_cmap():
    from matplotlib.colors import LinearSegmentedColormap
    return LinearSegmentedColormap.from_list("w", [INK3, "#f4f2ee", OURS])


# ----------------------------------------------------------------------------- figure 2
def fig2_main(runs, out):
    N = NUDGE["fig2"]
    main_run, noenv = os.path.join(runs, "shared_bip_car"), os.path.join(runs, "shared_no_env")
    S5 = load_strict(main_run)
    SF = load_dir(os.path.join(noenv, "readout_fixed_strict" + STRICT), "readout_")
    if not S5 or not SF:
        raise FileNotFoundError(2, "missing", os.path.join(main_run, "readout_combo5_strict" + STRICT))
    subs, (base, fixed, final) = paired(model_r(S5, "base"), model_r(SF, "combo_nosess"), model_r(S5, "combo_nosess"))
    mins = float(np.mean([d["fracs"]["0.2"].get("minutes", np.nan) for d in S5.values()]))
    nums = {"a": {"n": len(subs), "mean_base": base.mean(), "mean_fixed": fixed.mean(), "mean_final": final.mean(),
                  "final_minus_base": paired_stats(final, base), "minutes_20": mins}}

    fig = plt.figure(figsize=(W2, 80 * MM))
    gs = fig.add_gridspec(2, 2, width_ratios=[0.78, 1.0], height_ratios=[1.0, 1.0], wspace=0.34, hspace=0.78,
                          left=0.075, right=0.975, top=0.875, bottom=0.115)

    # ---------------- a: paired per-patient lines (tall, left)
    ax = fig.add_subplot(gs[:, 0])
    xs = np.array([0.0, 1.0, 2.0])
    for b, f, c in zip(base, fixed, final):
        ax.plot(xs, [b, f, c], color=PATIENT, lw=0.45, zorder=1)
        ax.scatter(xs, [b, f, c], s=2.2, color=PATIENT, zorder=2, linewidths=0)
    m = np.array([base.mean(), fixed.mean(), final.mean()])
    ax.plot(xs, m, color=OURS, lw=1.3, zorder=3)
    for x, v, fc in zip(xs, m, (BASE, FIXED, OURS)):
        ax.scatter([x], [v], s=17, color=fc, edgecolors="white", linewidths=0.6, zorder=4)
    # mean labels next to the points
    dx, dy = N["a_base_lab"]
    ax.text(xs[0] + dx, m[0] + dy, st.fmt(m[0]), ha="right", va="center", fontsize=6.5, color=BASE, fontweight="bold")
    dx, dy = N["a_fixed_lab"]
    ax.text(xs[1] + dx, m[1] + dy, st.fmt(m[1]), ha="center", va="bottom", fontsize=6.5, color=OURS)
    # the effect: bracket from the base level to the final level, right of the final point
    xb = xs[2] + 0.2
    ax.plot([xs[0], xb], [m[0], m[0]], color=RULE, lw=0.5, ls=(0, (2, 2)), zorder=0)
    ax.text(xs[2] + 0.11, m[2], st.fmt(m[2]), ha="left", va="bottom", fontsize=6.5, color=OURS, fontweight="bold")
    s_ = nums["a"]["final_minus_base"]
    st.bracket(ax, xb + 0.02, m[0], m[2] - 0.004, st.fmt(s_["mean_diff"], sign=True), color=ACCENT, w=0.07, pad=0.05)
    ax.text(xb + 0.14, (m[0] + m[2]) / 2 - 0.0125,
            f"[{st.fmt(s_['ci'][0])}, {st.fmt(s_['ci'][1])}]\n{s_['wins']}/{s_['n']} patients",
            fontsize=5.8, color=INK2, va="top", ha="left", linespacing=1.25, clip_on=False)
    ax.set_xticks(xs, ["lagged\nridge", "fixed\nuntrained bank", "final\nmodel"])
    ax.tick_params(axis="x", length=0, pad=3)
    ax.spines["bottom"].set_visible(False)
    ax.set_xlim(-0.42, 2.75)
    lo_, hi_ = min(base.min(), fixed.min(), final.min()), max(base.max(), fixed.max(), final.max())
    ax.set_ylim(lo_ - 0.012, hi_ + 0.008)
    ax.set_yticks(np.arange(np.ceil((lo_ - 0.012) * 20) / 20, hi_ + 0.008 + 1e-9, 0.05))
    ax.set_ylabel("test r, mean over contacts")
    st.range_frame(ax, x=False)
    st.headline(ax, "The final model is above ridge in every patient",
                tag=f"primary level · strict normalisation · {mins:.1f} min calibration")
    axa = ax

    # ---------------- b: all models (top right), session-wise normalisation
    T5 = load_dir(os.path.join(main_run, "readout_combo5"), "readout_")
    TF = load_dir(os.path.join(noenv, "readout_fixed"), "readout_")
    nfn = {s: d["models"]["nfn_mode"]["r"] for s, d in load_dir(os.path.join(runs, "nfn_baseline"), "nfn_").items()}
    rows = [("final model", model_r(T5, "combo"), OURS), (st.LBL_FIXED, model_r(TF, "combo"), FIXED),
            ("lagged ridge", model_r(T5, "base"), BASE), ("LeadFormer", find_leadformer(runs), BASE),
            ("NeuroFlowNet", nfn, BASE)]
    rows = [r for r in rows if r[1]]
    ax = fig.add_subplot(gs[0, 1])
    yy = np.arange(len(rows))[::-1].astype(float)
    ax.axhspan(yy[1] - 0.5, yy[0] + 0.5, color=WASH, lw=0, zorder=0)        # our two rows
    nums["b"] = {}
    for y, (lab, d, col) in zip(yy, rows):
        v = np.array(list(d.values()), float)
        off = st.beeswarm(v, size_data=0.0065, max_off=0.30)
        dot = col if col != BASE else "#b3aea5"
        ax.scatter(v, y + off, s=2.6, color=dot, linewidths=0, zorder=2, alpha=0.95)
        lo, hi = st.boot_ci(v)
        mc = OURS if col in (OURS, FIXED) else INK2
        ax.plot([lo, hi], [y, y], color=mc, lw=1.0, zorder=3)
        ax.scatter([v.mean()], [y], s=13, marker="D", color=col if col != BASE else INK2, edgecolors="white",
                   linewidths=0.5, zorder=4)
        ax.text(max(hi, v.max()) + 0.012, y, st.fmt(v.mean()), va="center", ha="left", fontsize=6.2, color=mc,
                fontweight="bold" if lab == "final model" else "normal")
        nums["b"][lab] = {"n": len(v), "mean": float(v.mean()), "ci": [lo, hi]}
    ax.set_yticks(yy, [r[0] for r in rows])
    for t, r in zip(ax.get_yticklabels(), rows):
        t.set_color(OURS if r[2] in (OURS, FIXED) else INK)
    st.bare_y(ax)
    ax.set_ylim(yy[-1] - 0.6, yy[0] + 0.6)
    ax.set_xlim(0, 0.42)
    ax.set_xticks([0, 0.1, 0.2, 0.3, 0.4])
    ax.set_xlabel("test r per patient")
    if "lagged ridge" in nums["b"]:
        st.ref_line(ax, nums["b"]["lagged ridge"]["mean"], axis="x")
    st.range_frame(ax, y=False)
    st.headline(ax, "Above every baseline on the same trials",
                tag="secondary level · session-wise normalisation · diamond = mean, bar = 95 % CI")
    axb = ax

    # ---------------- c: calibration efficiency (bottom right), primary level, strict
    ax = fig.add_subplot(gs[1, 1])
    fr = [f for f in ("0.05", "0.1", "0.2") if all(f in d["fracs"] for d in S5.values())]
    xm = np.array([np.mean([d["fracs"][f]["minutes"] for d in S5.values()]) for f in fr])
    Y = {k: np.array([[d["fracs"][f]["models"][k]["r"] for f in fr] for d in S5.values()]) for k in ("base", "combo_nosess")}
    nums["c"] = {"minutes": xm.tolist()}
    cdx = N["c_dx"]
    for k, col, lab, sh in (("base", BASE, "lagged ridge", -cdx), ("combo_nosess", OURS, "final model", cdx)):
        mu = Y[k].mean(0)
        ci = np.array([st.boot_ci(Y[k][:, j], 4000) for j in range(len(fr))])
        # CI as thin whiskers, nudged apart horizontally so the two models never sit on top of each other
        for x, (l, h) in zip(xm + sh, ci):
            ax.plot([x, x], [l, h], color=col, lw=0.55, alpha=0.75, solid_capstyle="butt", zorder=2)
        ax.plot(xm + sh, mu, color=col, lw=1.1, zorder=3)
        ax.scatter(xm + sh, mu, s=10, color=col, edgecolors="white", linewidths=0.5, zorder=4)
        ax.text(xm[-1] + 0.28, mu[-1], lab, color=col, fontsize=6.2, va="center", ha="left")
        nums["c"][lab] = mu.tolist()
        nums["c"][lab + "_ci"] = ci.tolist()
    note_top = -np.inf
    if "0.1" in fr and "0.2" in fr:
        j10, j20 = fr.index("0.1"), fr.index("0.2")
        yb20 = Y["base"][:, j20].mean()
        ax.plot([xm[0] - 0.25, xm[j20] - cdx], [yb20, yb20], color=RULE, lw=0.5, ls=(0, (2, 2)), zorder=0)
        s_ = paired_stats(Y["combo_nosess"][:, j10], Y["base"][:, j20])
        nums["c"]["final10_minus_base20"] = s_
        y10 = Y["combo_nosess"][:, j10].mean()
        x10 = xm[j10] + cdx
        ax.scatter([x10], [y10], s=48, facecolors="none", edgecolors=INK, linewidths=0.6, zorder=5)
        dx, dy = N["c_note"]
        note_top = y10 + dy + 0.027
        ax.annotate(f"final {xm[j10]:.1f} min − ridge {xm[j20]:.1f} min\n"
                    f"{st.fmt(s_['mean_diff'], sign=True)} [{st.fmt(s_['ci'][0])}, {st.fmt(s_['ci'][1])}], "
                    f"{s_['wins']}/{s_['n']}",
                    xy=(x10, y10), xytext=(x10 + dx, y10 + dy),
                    fontsize=5.8, color=INK, ha="left", va="bottom", linespacing=1.3,
                    arrowprops=dict(arrowstyle="-", color=INK, lw=0.45, shrinkA=0.5, shrinkB=3.2,
                                    connectionstyle="angle,angleA=0,angleB=90,rad=0"))
    ax.set_xticks(xm, [f"{v:.1f}" for v in xm])
    ax.set_xlim(xm[0] - 0.4, xm[-1] + 1.9)
    allci = np.concatenate([np.ravel(nums["c"][k]) for k in nums["c"] if k.endswith("_ci")])
    lo_, hi_ = allci.min(), allci.max()
    ax.set_ylim(lo_ - 0.006, max(hi_, note_top) + 0.004)
    ax.set_yticks(np.round(np.arange(np.ceil(lo_ * 25) / 25, hi_ + 1e-9, 0.04), 3))
    ax.set_xlabel("calibration (min)")
    ax.set_ylabel("test r")
    st.range_frame(ax)
    st.headline(ax, "Half the calibration already beats ridge",
                tag="primary level · strict · same test trials · ds004752 only (not replicated, Fig. 7)")
    axc = ax

    st.letters(fig, [(axa, "a", -1.5), (axb, "b", -1.5), (axc, "c", -1.5)])
    st.save(fig, out, "fig2_main")
    dump_numbers(out, "fig2_main", nums)


FIGS = {"fig1": fig1_scene, "fig2": fig2_main}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--only", nargs="*", default=None)
    ap.add_argument("--style", default="own", choices=["own", "sp"], help="sp = SciencePlots base style")
    a = ap.parse_args()
    st.setup(a.style)
    from s2i import paperfigs_b
    FIGS.update(paperfigs_b.FIGS)
    for k, fn in FIGS.items():
        if a.only and k not in a.only:
            continue
        print(k)
        try:
            fn(a.runs, a.out)
        except FileNotFoundError as e:
            print(f"  skipped: missing {e.filename}")
        except Exception:                                  # keep going; the traceback says what to fix
            import traceback
            traceback.print_exc()
            print(f"  FAILED: {k}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/paperfigs_b.py
"""
Paper figures 3-7 and S1 (Phase 10 rework, 12 Oct 2026). Style: s2i/figstyle.py; helpers and loaders: s2i/paperfigs.py.
Registered in paperfigs.main(), so `python -m s2i.paperfigs --runs RUNS --out OUT --only fig3` renders them.

  fig3_transfer   a gain decomposition, primary level, strict (ridge -> fixed bank -> 3 fixed -> 3 learned -> final)
                  b frequency response of the shared filter bank, initial vs learned; c peak frequency per pair
  fig4_reliab     a calibration K-fold r vs test r per contact (primary, strict), per-patient Spearman in the inset
                  b coverage (primary, strict); b' top quarter - patient mean; c longevity (secondary, session-wise)
  fig5_power      band power without intracranial calibration (secondary): a per band, b zero-shot vs 4-min ridge
  fig6_protocol   a one factor at a time; b literature-inspired emulations; c per-patient target + reference effect
  fig7_external   a H1 per patient (ScaleOLDNEW); b each finding in both cohorts; c common mode vs interaction
  figS1_chanqc    ScaleOLDNEW channel audit (post hoc): raw SD vs |r| with inputs without a 10-05 name

Every figure writes <name>_numbers.json with each number printed on it.
"""
import colorsys
import glob
import json
import os

import numpy as np

from s2i import figstyle as st
from s2i.figstyle import (ACCENT, ACCENT_SOFT, BASE, EXT, FIXED, INK, INK2, INK3, MM, OURS, OURS_SOFT, PATIENT, RULE,
                          W2, plt)
from s2i.paperfigs import STRICT, dump_numbers, load_dir, load_strict, model_r, paired, paired_stats

# hand-tuned label offsets; edit after looking at the real-data render
NUDGE = {
    "fig3": {"total_x": 4.95},
    "fig4": {},
    "fig5": {"lab_gap": 0.0135},
    "fig6": {},
    "fig7": {},
}
P17 = "p17"


def _runs(runs):
    return os.path.join(runs, "shared_bip_car"), os.path.join(runs, "shared_no_env")


def _need(D, path):
    if not D:
        raise FileNotFoundError(2, "missing", path)
    return D


def _spread(ys, gap):
    """Push label positions apart (keeps order, minimal moves) so neighbours are at least `gap` apart."""
    o = np.argsort(ys)
    y = np.asarray(ys, float)[o].copy()
    for _ in range(50):
        moved = False
        for i in range(1, len(y)):
            if y[i] - y[i - 1] < gap:
                d = (gap - (y[i] - y[i - 1])) / 2
                y[i - 1] -= d
                y[i] += d
                moved = True
        if not moved:
            break
    out = np.empty_like(y)
    out[o] = y
    return out


def _patient_colours(n):
    return [colorsys.hls_to_rgb((0.04 + i / n) % 1, 0.66, 0.38) for i in range(n)]


def _short(s):
    return s.replace("sub-", "")


# ============================================================================= figure 3
def _bank_responses(main_run, nfft=4096, fs=200):
    """(f, H0 initial, Hs learned per fold) of the quadrature pairs, peak-normalised. Learned banks are read from the
    encoder checkpoints (torch); without torch a cached bank_responses.npz in the same folder is used."""
    encs = sorted(glob.glob(os.path.join(main_run, "enc_sub-*.pt")))
    try:
        import torch
        from s2i.shared import TemporalEncoder
    except ImportError:
        torch = None
    if encs and torch is not None:
        def resp(W, n):
            f = np.fft.rfftfreq(nfft, 1 / fs)
            H = np.sqrt(np.abs(np.fft.rfft(W[0:2 * n:2], nfft)) ** 2 + np.abs(np.fft.rfft(W[1:2 * n:2], nfft)) ** 2)
            return f, H / H.max(1, keepdims=True)
        cfg = torch.load(encs[0], map_location="cpu", weights_only=False)["cfg"]
        n = cfg["n_pairs"]
        f, H0 = resp(TemporalEncoder(**cfg).conv.weight.detach().numpy()[:, 0], n)
        Hs = [resp(torch.load(p, map_location="cpu", weights_only=False)["state"]["conv.weight"].numpy()[:, 0], n)[1]
              for p in encs]
        return f, H0, np.array(Hs)
    cache = os.path.join(main_run, "bank_responses.npz")
    if os.path.exists(cache):
        z = np.load(cache)
        return z["f"], z["H0"], z["Hs"]
    raise FileNotFoundError(2, "missing", os.path.join(main_run, "enc_sub-*.pt"))


def fig3_transfer(runs, out):
    N = NUDGE["fig3"]
    main_run, noenv = _runs(runs)
    S5 = _need(load_strict(main_run), main_run)
    L = lambda name: load_dir(os.path.join(noenv, name + STRICT), "readout_")   # noqa: E731
    SF, F3, L3 = L("readout_fixed_strict"), L("readout_fixed_arch3_strict"), L("readout_learned_arch3_strict")
    _need(F3, os.path.join(noenv, "readout_fixed_arch3_strict" + STRICT))
    steps = [("lagged\nridge", model_r(S5, "base"), BASE), ("fixed\nbank", model_r(SF, "combo_nosess"), FIXED),
             ("3 fixed\nbanks", model_r(F3, "combo_nosess"), FIXED), ("3 learned\nbanks", model_r(L3, "combo_nosess"), ACCENT),
             ("final\nmodel", model_r(S5, "combo_nosess"), OURS)]
    subs, V = paired(*[s[1] for s in steps])
    m = np.array([v.mean() for v in V])
    mins = float(np.mean([S5[s]["fracs"]["0.2"].get("minutes", np.nan) for s in subs]))
    nums = {"a": {"n": len(subs), "minutes": mins, "means": dict(zip([s[0].replace("\n", " ") for s in steps], m.tolist())),
                  "steps": {}, "final_minus_base": paired_stats(V[-1], V[0])}}

    fig = plt.figure(figsize=(W2, 72 * MM))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 1.08], height_ratios=[1.55, 1.0], wspace=0.24, hspace=0.62,
                          left=0.075, right=0.985, top=0.875, bottom=0.12)

    # ---------------- a: waterfall; bar = mean step, dots = previous mean + each patient's step
    ax = fig.add_subplot(gs[:, 0])
    w_ = 0.58
    ax.plot([-w_ / 2, w_ / 2], [m[0], m[0]], color=BASE, lw=2.4, solid_capstyle="butt", zorder=3)
    ax.text(-w_ / 2 - 0.08, m[0], st.fmt(m[0]), ha="right", va="center", fontsize=6.3, color=BASE, fontweight="bold")
    for k in range(1, len(steps)):
        lab, _, col = steps[k]
        d = V[k] - V[k - 1]
        y0, y1 = m[k - 1], m[k]
        ax.add_patch(plt.Rectangle((k - w_ / 2, min(y0, y1)), w_, abs(y1 - y0), fc=col, ec="none", zorder=2,
                                   alpha=0.9 if col != FIXED else 1.0))
        ax.plot([k - 1 + w_ / 2, k - w_ / 2], [y0, y0], color=INK3, lw=0.45, ls=(0, (1.5, 1.5)), zorder=1)
        off = st.beeswarm(y0 + d, size_data=0.0012, max_off=0.24)
        dc = INK if col == ACCENT else (OURS if col in (FIXED, OURS) else INK2)
        ax.scatter(k + off, y0 + d, s=3.2, color="white", edgecolors=dc, linewidths=0.45, zorder=4)
        s_ = paired_stats(V[k], V[k - 1])
        nums["a"]["steps"][lab.replace("\n", " ")] = s_
        top = max(y1, (y0 + d).max())
        hot = col == ACCENT
        ax.text(k, top + 0.0022, st.fmt(s_["mean_diff"], sign=True), ha="center", va="bottom", fontsize=6.6 if hot else 6.2,
                color=ACCENT if hot else (OURS if col != BASE else INK2), fontweight="bold" if hot else "normal")
        ax.text(k, top + 0.0022 + 0.0042, f"{s_['wins']}/{s_['n']}", ha="center", va="bottom", fontsize=5.6, color=INK3)
    ax.text(len(steps) - 1 + w_ / 2 + 0.08, m[-1], st.fmt(m[-1]), ha="left", va="center", fontsize=6.3, color=OURS,
            fontweight="bold")
    xt = N["total_x"]
    s_ = nums["a"]["final_minus_base"]
    ax.plot([0 + w_ / 2, xt], [m[0], m[0]], color=RULE, lw=0.5, ls=(0, (2, 2)), zorder=0)
    st.bracket(ax, xt, m[0], m[-1], st.fmt(s_["mean_diff"], sign=True), color=INK, w=0.08, pad=0.06, fs=6.4)
    ax.set_xticks(range(len(steps)), [s[0] for s in steps])
    for t, s in zip(ax.get_xticklabels(), steps):
        t.set_color(ACCENT if s[2] == ACCENT else INK)
    ax.tick_params(axis="x", length=0, pad=3)
    ax.spines["bottom"].set_visible(False)
    lo = min(m[0], min((V[k] - V[k - 1] + m[k - 1]).min() for k in range(1, len(steps))))
    hi = max(m.max(), max((V[k] - V[k - 1] + m[k - 1]).max() for k in range(1, len(steps)))) + 0.011
    ax.set_xlim(-0.75, xt + 0.55)
    ax.set_ylim(lo - 0.004, hi)
    ax.set_yticks(np.round(np.arange(np.ceil((lo - 0.004) * 100) / 100, hi + 1e-9, 0.01), 2))
    ax.set_ylabel("test r, mean over patients")
    st.range_frame(ax, x=False)
    st.headline(ax, "Learning the filters adds to a fixed bank",
                tag=f"primary level · strict · {mins:.1f} min calibration · dots = patients")
    axa = ax

    # ---------------- b: frequency response; c: peak frequency per pair (shared log-frequency axis)
    f, H0, Hs = _bank_responses(main_run)
    H1 = Hs.mean(0)
    n = H0.shape[0]
    sel = (f >= 1.5) & (f <= 80)
    axb = fig.add_subplot(gs[0, 1])
    for k in range(n):
        axb.fill_between(f[sel], H0[k, sel], H1[k, sel], color=ACCENT, alpha=0.10, lw=0, zorder=1)
        axb.plot(f[sel], H0[k, sel], color=INK3, lw=0.5, ls=(0, (2.2, 1.6)), zorder=2)
        axb.plot(f[sel], H1[k, sel], color=OURS, lw=0.8, zorder=3)
    axb.set_xscale("log")
    axb.set_xlim(1.5, 80)
    axb.set_ylim(0, 1.04)
    axb.set_yticks([0, 0.5, 1.0])
    axb.set_ylabel("gain")
    axb.tick_params(axis="x", which="both", labelbottom=False)
    st.range_frame(axb, x=False)
    axb.spines["bottom"].set_visible(False)
    st.headline(axb, "Shared filter bank, before and after learning",
                tag=f"dashed = initial · solid = learned, mean of {len(Hs)} folds · {n} quadrature pairs")

    axc = fig.add_subplot(gs[1, 1], sharex=axb)
    pk0 = f[np.argmax(H0 * sel, 1)]
    pk = np.array([f[np.argmax(H * sel, 1)] for H in Hs])
    sh = np.log2(pk / pk0[None])                          # shift of each pair's peak, in octaves
    med, plo, phi = np.median(sh, 0), np.percentile(sh, 10, 0), np.percentile(sh, 90, 0)
    st.ref_line(axc, 0)
    for k in range(n):
        axc.plot([pk0[k], pk0[k]], [0, med[k]], color=ACCENT, lw=1.1, alpha=0.75, zorder=1, solid_capstyle="butt")
        axc.plot([pk0[k] * 1.035] * 2, [plo[k], phi[k]], color=OURS, lw=0.5, alpha=0.7, zorder=2)
    axc.scatter(pk0, med, s=9, color=OURS, edgecolors="white", linewidths=0.4, zorder=4)
    axc.set_xscale("log")
    axc.set_xticks([2, 4, 8, 16, 32, 64], ["2", "4", "8", "16", "32", "64"])
    axc.minorticks_off()
    axc.set_xlim(1.5, 80)
    m_ = max(0.25, np.abs(np.r_[plo, phi]).max() * 1.05)
    yt = 0.25 if m_ < 0.6 else 0.5
    m_ = np.ceil(m_ / yt - 1e-9) * yt                     # whiskers end inside the range frame
    axc.set_ylim(-m_ * 1.04, m_ * 1.04)
    axc.set_yticks(np.arange(-m_, m_ + 1e-9, yt))
    axc.set_ylabel("peak shift (oct.)")
    axc.set_xlabel("frequency (Hz)")
    st.range_frame(axc)
    st.headline(axc, "Peak shift per pair", tag="median and 10–90 % of folds, at the initial peak frequency")
    nums["b"] = {"n_pairs": int(n), "n_folds": int(len(Hs))}
    nums["c"] = {"peak_initial_hz": pk0.tolist(), "peak_shift_oct_median": med.tolist()}

    st.letters(fig, [(axa, "a", -1.5), (axb, "b", -1.5), (axc, "c", -1.5)])
    st.save(fig, out, "fig3_transfer")
    dump_numbers(out, "fig3_transfer", nums)


# ============================================================================= figure 4
def _mixed_slope(rows):
    """Slope of r on session gap with a random intercept per patient (statsmodels); OLS if unavailable."""
    G = np.array([r[1] for r in rows], float)
    V = np.array([r[2] for r in rows], float)
    try:
        import pandas as pd
        import statsmodels.formula.api as smf
        df = pd.DataFrame(rows, columns=["subject", "gap", "r"])
        fit = smf.mixedlm("r ~ gap", df, groups=df["subject"]).fit()
        ci = fit.conf_int().loc["gap"].tolist()
        return float(fit.params["gap"]), float(fit.params["Intercept"]), ci, "mixed"
    except Exception:
        s, i = np.polyfit(G, V, 1)
        return float(s), float(i), [np.nan, np.nan], "ols"


def fig4_reliab(runs, out):
    from scipy.stats import spearmanr
    main_run, _ = _runs(runs)
    S5 = _need(load_strict(main_run), main_run)
    subs = sorted(S5)
    cols = _patient_colours(len(subs))
    per = {}
    for s in subs:
        mm = S5[s]["fracs"]["0.2"]["models"]["combo_nosess"]
        cv, te = np.array(mm["cv_contacts"], float), np.array(mm["contacts"], float)
        ok = np.isfinite(cv) & np.isfinite(te)
        per[s] = (cv[ok], te[ok])
    rho = np.array([spearmanr(*per[s]).correlation for s in subs])
    cv = np.concatenate([per[s][0] for s in subs])
    te = np.concatenate([per[s][1] for s in subs])
    nums = {"a": {"n_contacts": int(len(cv)), "rho_per_patient": dict(zip(subs, rho.tolist())),
                  "rho_median": float(np.median(rho)), "rho_mean": float(rho.mean()),
                  "rho_range": [float(rho.min()), float(rho.max())],
                  "rho_pooled_reference_only": float(spearmanr(cv, te).correlation)}}

    fig = plt.figure(figsize=(W2, 92 * MM))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.42, 0.5, 0.92], height_ratios=[1.0, 1.0], wspace=0.42, hspace=0.72,
                          left=0.07, right=0.985, top=0.905, bottom=0.095)

    # ---------------- a: square scatter, light per-patient colours
    ax = fig.add_subplot(gs[:, 0])
    lim = (np.floor(min(np.percentile(cv, 0.3), np.percentile(te, 0.3)) * 10) / 10,
           np.ceil(max(np.percentile(cv, 99.8), np.percentile(te, 99.8)) * 10) / 10)
    ax.plot(lim, lim, color=RULE, lw=0.5, zorder=0)
    for s, c in zip(subs, cols):
        ax.scatter(*per[s], s=3.2, color=c, alpha=0.75, linewidths=0, zorder=2)
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_aspect("equal")
    ax.set_anchor("N")
    tk = np.round(np.arange(lim[0], lim[1] + 1e-9, 0.2), 1)
    ax.set_xticks(tk)
    ax.set_yticks(tk)
    ax.set_xlabel("calibration K-fold r (before use)")
    ax.set_ylabel("test r (later trials)")
    st.range_frame(ax)
    ax.text(0.04, 0.965, f"ρ = {np.median(rho):.2f}", transform=ax.transAxes, fontsize=9, color=OURS,
            fontweight="bold", va="top", ha="left")
    ax.text(0.04, 0.885, "median per patient", transform=ax.transAxes, fontsize=5.8, color=INK2, va="top", ha="left")
    # inset: one dot per patient on a 0-1 rho scale (lower right is empty in a correlated scatter)
    ins = ax.inset_axes([0.55, 0.08, 0.42, 0.13])
    off = st.beeswarm(rho, size_data=0.03, max_off=0.42)
    ins.scatter(rho, off, s=6, c=cols, linewidths=0.3, edgecolors="white", zorder=2)
    ins.plot([np.median(rho)] * 2, [-0.55, 0.55], color=OURS, lw=1.0, zorder=3)
    ins.set_xlim(0, 1)
    ins.set_ylim(-0.6, 0.6)
    ins.set_xticks([0, 0.5, 1], ["0", "0.5", "1"])
    ins.tick_params(labelsize=5.4, length=1.5, pad=1)
    ins.set_yticks([])
    for sp in ("left", "top", "right"):
        ins.spines[sp].set_visible(False)
    ins.patch.set_alpha(0)
    ins.set_title("ρ per patient", fontsize=5.6, color=INK2, pad=1.5, loc="left")
    st.headline(ax, "Calibration predicts which contacts work",
                tag=f"final model · primary level · strict · {len(cv)} contacts, 15 colours = patients")
    axa = ax

    # ---------------- b: coverage curve (pooled contacts, most reliable first)
    ax = fig.add_subplot(gs[0, 1:])
    order = np.argsort(-cv)
    kept = np.arange(1, len(cv) + 1) / len(cv) * 100
    cum = np.cumsum(te[order]) / np.arange(1, len(cv) + 1)
    k0 = 10
    ax.plot(kept[k0:], cum[k0:], color=OURS, lw=1.3, zorder=3)
    ax.fill_between(kept[k0:], te.mean(), cum[k0:], color=OURS_SOFT, lw=0, zorder=1)
    st.ref_line(ax, te.mean())
    ax.text(52, te.mean() - 0.005, f"all contacts {st.fmt(te.mean())}", ha="center", va="top", fontsize=5.8,
            color=INK2)
    nums["b"] = {"all_contacts_mean": float(te.mean()), "thresholds": {}}
    for thr in (0.3, 0.4, 0.5):
        sel = cv > thr
        if sel.sum() < 10:
            continue
        x, y = sel.mean() * 100, te[sel].mean()
        ax.scatter([x], [y], s=13, color=ACCENT, zorder=4, edgecolors="white", linewidths=0.5)
        ax.text(x + 1.6, y + 0.004, f"> {thr:.1f}", fontsize=5.8, color=ACCENT, va="bottom", ha="left")
        nums["b"]["thresholds"][str(thr)] = {"kept_pct": float(x), "mean_test_r": float(y)}
    ax.set_xlim(0, 100)
    ax.set_ylim(te.mean() - 0.03, cum[k0:].max() + 0.02)
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_xlabel("contacts kept (%), most reliable first")
    ax.set_ylabel("mean test r")
    st.range_frame(ax)
    st.headline(ax, "Keeping reliable contacts raises accuracy", tag="orange = calibration r threshold")
    axb = ax

    # ---------------- b': top quarter (by calibration r) minus the patient's mean, per patient
    ax = fig.add_subplot(gs[1, 1])
    dq = np.array([per[s][1][per[s][0] >= np.percentile(per[s][0], 75)].mean() - per[s][1].mean() for s in subs])
    off = st.beeswarm(dq, size_data=0.008, max_off=0.34)
    ax.scatter(off, dq, s=6, c=cols, linewidths=0.3, edgecolors="white", zorder=2)
    lo, hi = st.boot_ci(dq)
    ax.plot([0.5, 0.5], [lo, hi], color=OURS, lw=1.0, zorder=3)
    ax.scatter([0.5], [dq.mean()], s=14, marker="D", color=OURS, edgecolors="white", linewidths=0.5, zorder=4)
    ax.text(0.62, dq.mean(), st.fmt(dq.mean(), sign=True), fontsize=6.2, color=OURS, fontweight="bold", va="center")
    ax.text(0.62, dq.mean() - 0.022, f"{int((dq > 0).sum())}/{len(dq)}", fontsize=5.6, color=INK3, va="center")
    st.ref_line(ax, 0)
    ax.set_xlim(-0.5, 1.2)
    ax.set_xticks([])
    ax.spines["bottom"].set_visible(False)
    ax.set_ylim(min(0, dq.min()) - 0.01, dq.max() + 0.015)
    ax.set_ylabel("Δ test r")
    st.range_frame(ax, x=False)
    st.headline(ax, "Top quarter", tag="vs patient mean")
    nums["b_prime"] = {"top_quarter_minus_mean": paired_stats(dq, np.zeros_like(dq))}
    axb2 = ax

    # ---------------- c: longevity across sessions (secondary level, session-wise normalisation)
    ax = fig.add_subplot(gs[1, 2])
    lon = load_dir(os.path.join(main_run, "readout_longevity"), "readout_")
    rows, rows_adv = [], []
    for d in lon.values():
        fr, order_ = d["fracs"]["0.2"], d["session_order"]
        last = order_.index(fr["calib_sessions"][-1])
        mods = fr["models"]
        pts = []
        for s, v in mods["combo"].get("sessions", {}).items():
            if d["test_trials_per_session"].get(s, 0) >= 5:
                pts.append((order_.index(s) - last, v, mods.get("base", {}).get("sessions", {}).get(s)))
        pts.sort()
        if len(pts) >= 2:
            ax.plot([p[0] for p in pts], [p[1] for p in pts], color=PATIENT, lw=0.5, zorder=1)
        rows += [(d["subject"], p[0], p[1]) for p in pts]
        rows_adv += [(d["subject"], p[0], p[1] - p[2]) for p in pts if p[2] is not None]
    nums["c"] = {"n_patient_sessions": len(rows)}
    if rows:
        slope, icpt, ci, kind = _mixed_slope(rows)
        G = np.array([r[1] for r in rows])
        gx = np.array([G.min(), G.max()])
        ax.plot(gx, icpt + slope * gx, color=OURS, lw=1.5, zorder=3)
        ax.text(gx[1] + 0.12, icpt + slope * gx[1], f"{st.fmt(slope, sign=True)}\nper session", ha="left", va="center",
                fontsize=6.0, color=OURS, fontweight="bold", linespacing=1.15)
        nums["c"].update({"slope": slope, "slope_ci": ci, "model": kind})
        if rows_adv:
            sa, _, cia, _ = _mixed_slope(rows_adv)
            nums["c"].update({"advantage_slope": sa, "advantage_slope_ci": cia})
        ax.set_xticks(range(int(G.min()), int(G.max()) + 1))
        ax.set_xlim(G.min() - 0.3, G.max() + 1.6)
    ax.set_xlabel("sessions after calibration")
    ax.set_ylabel("test r")
    st.range_frame(ax)
    st.headline(ax, "Accuracy drifts across sessions", tag="secondary level · lines = patients")
    axc = ax

    st.letters(fig, [(axa, "a", -1.5), (axb, "b", -1.5), (axb2, "c", -1.5), (axc, "d", -1.5)])
    st.save(fig, out, "fig4_reliability")
    dump_numbers(out, "fig4_reliability", nums)


# ============================================================================= figure 5
BAND_SYM = {"delta": "δ", "theta": "θ", "alpha": "α", "beta": "β"}


def fig5_power(runs, out):
    N = NUDGE["fig5"]
    p = os.path.join(runs, "zeroshot_power", "zeroshot_subjects.json")
    if not os.path.exists(p):
        raise FileNotFoundError(2, "missing", p)
    zs = json.load(open(p))
    bands = list(zs[0]["bands"])
    # (key, label, colour, line style, marker, emphasis)
    rows = [("zs_global", "global scalp average", "#b9b4ab", (0, (1.2, 1.4)), "^", 0),
            ("zs_nearest", "nearest scalp channel", "#a8a39a", (0, (1.2, 1.4)), "s", 0),
            ("zs_geo_shuf", "positions shuffled", "#8f8a82", (0, (3, 1.6)), "o", 0),
            ("cal_ridge", "ridge, 4 min SEEG", BASE, "-", "o", 1),
            ("zs_geo", "zero-shot, no SEEG", ACCENT, "-", "D", 2),
            ("zs_cal", "zero-shot + ridge", OURS, "-", "o", 1)]
    get = lambda m, b: np.array([r["bands"][b][m]["r_induced"] for r in zs], float)   # noqa: E731
    fig = plt.figure(figsize=(W2, 66 * MM))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.9, 1.0], wspace=0.3, left=0.065, right=0.985, top=0.85, bottom=0.15)

    ax = fig.add_subplot(gs[0])
    xb = np.arange(len(bands), dtype=float)
    nums = {"a": {}, "n": len(zs)}
    # patients behind the zero-shot line
    for bi, b in enumerate(bands):
        v = get("zs_geo", b)
        off = st.beeswarm(v, size_data=0.006, max_off=0.2)
        ax.scatter(bi + off * 0.6, v, s=2.4, color=ACCENT_SOFT, edgecolors="#e2b7a6", linewidths=0.3, zorder=1)
    ends = []
    dodge = np.linspace(-0.075, 0.075, len(rows))
    for (m, lab, col, ls, mk, emph), dx in zip(rows, dodge):
        mu = np.array([get(m, b).mean() for b in bands])
        ci = np.array([st.boot_ci(get(m, b), 4000) for b in bands])
        ax.plot(xb + dx, mu, color=col, lw=(0.7, 0.9, 1.5)[emph], ls=ls, zorder=2 + emph)
        for x, (l_, h_) in zip(xb + dx, ci):
            ax.plot([x, x], [l_, h_], color=col, lw=0.5, alpha=0.8, zorder=2 + emph, solid_capstyle="butt")
        ax.scatter(xb + dx, mu, s=(9, 11, 16)[emph], marker=mk, color=col if emph else "white", edgecolors=col,
                   linewidths=0.7, zorder=3 + emph)
        ends.append(mu[-1])
        nums["a"][m] = {b: {"mean": float(u), "ci": c.tolist()} for b, u, c in zip(bands, mu, ci)}
    ly = _spread(ends, N["lab_gap"])
    for (m, lab, col, ls, mk, emph), y0, y1 in zip(rows, ends, ly):
        ax.plot([xb[-1] + 0.12, xb[-1] + 0.2], [y0, y1], color=col, lw=0.4, zorder=1)
        ax.text(xb[-1] + 0.23, y1, lab, color=col if emph else INK2, fontsize=6.0, va="center", ha="left",
                fontweight="bold" if emph == 2 else "normal")
    ax.set_xticks(xb, [BAND_SYM.get(b, b) for b in bands], fontsize=8)
    ax.tick_params(axis="x", length=0, pad=3)
    ax.spines["bottom"].set_visible(False)
    ax.set_xlim(-0.3, len(bands) - 1 + 1.25)
    ax.set_ylabel("r of log band power (induced)")
    ax.set_ylim(bottom=max(0, ax.get_ylim()[0]))
    st.range_frame(ax, x=False)
    st.headline(ax, "Band power of a new patient's contacts, without their SEEG",
                tag="secondary level · leave one patient out · mean and 95 % CI · dots = patients (zero-shot)")
    axa = ax

    # ---------------- b: zero-shot vs 4-min ridge per patient (mean over bands)
    ax = fig.add_subplot(gs[1])
    zg = np.mean([get("zs_geo", b) for b in bands], 0)
    cr = np.mean([get("cal_ridge", b) for b in bands], 0)
    lim = (np.floor(min(zg.min(), cr.min()) * 20) / 20, np.ceil(max(zg.max(), cr.max()) * 20) / 20)
    ax.plot(lim, lim, color=RULE, lw=0.5, zorder=0)
    ax.scatter(cr, zg, s=16, color=ACCENT, edgecolors="white", linewidths=0.6, zorder=3)
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_aspect("equal")
    ax.set_anchor("N")
    tk = np.round(np.arange(lim[0], lim[1] + 1e-9, 0.05), 2)
    ax.set_xticks(tk)
    ax.set_yticks(tk)
    ax.set_xlabel("ridge, 4 min of the patient's SEEG")
    ax.set_ylabel("zero-shot, no SEEG")
    st.range_frame(ax)
    above = int((zg > cr).sum())
    ax.text(0.05, 0.95, f"{above}/{len(zg)} above", transform=ax.transAxes, fontsize=6.0, color=ACCENT, va="top")
    st.headline(ax, "Comparable to 4 min of calibration", tag="per patient · mean over the four bands")
    nums["b"] = {"zero_shot_mean": float(zg.mean()), "ridge_mean": float(cr.mean()), "zero_shot_above": above}
    axb = ax

    st.letters(fig, [(axa, "a", -1.5), (axb, "b", -1.5)])
    st.save(fig, out, "fig5_band_power")
    dump_numbers(out, "fig5_band_power", nums)


# ============================================================================= figure 6
ONE_FACTOR = [   # protocol.py label -> short label (what was changed relative to our protocol)
    ("split: random vs chronological (20 % calibration)", "random split, 20 %"),
    ("split: random vs chronological (90 % training)", "random split, 90 %"),
    ("amount: 90 % vs 20 % training, both chronological", "90 % training"),
    ("units: random 1-s segments vs random trials (80 %)", "1-s segments"),
    ("target: monopolar vs bipolar (CAR scalp, clean split)", "monopolar target"),
    ("scalp reference: original vs CAR (bipolar target, clean split)", "original scalp reference"),
    ("target + reference: monopolar/original vs bipolar/CAR", "target + reference"),
    ("metric: NeuroFlowNet-style vs ours (same predictions, clean)", "flattened metric"),
]
KEY_ROW = "target + reference"


def fig6_protocol(runs, out):
    pdir = os.path.join(runs, "protocol")
    pj = os.path.join(pdir, "protocol.json")
    if not os.path.exists(pj):
        raise FileNotFoundError(2, "missing", pj)
    summ = json.load(open(pj))
    per = load_dir(pdir, "protocol_")
    n = len(per) or 15
    fig = plt.figure(figsize=(W2, 82 * MM))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.12, 1.0], height_ratios=[1.0, 1.08], wspace=0.5, hspace=0.78,
                          left=0.15, right=0.975, top=0.89, bottom=0.1)

    # ---------------- a: one factor at a time
    ax = fig.add_subplot(gs[:, 0])
    rows = [(lab, summ[f"one_factor/{k}/enc"]) for k, lab in ONE_FACTOR if f"one_factor/{k}/enc" in summ]
    y = np.arange(len(rows))[::-1].astype(float)
    nums = {"a": {}}
    for yi, (lab, r) in zip(y, rows):
        key = lab == KEY_ROW
        col = ACCENT if key else OURS
        lo, hi = r["ci"]
        sig = lo > 0 or hi < 0
        ax.plot([lo, hi], [yi, yi], color=col, lw=1.6 if key else 1.0, solid_capstyle="butt", zorder=2)
        ax.scatter([r["diff"]], [yi], s=30 if key else 18, zorder=3, color=col if sig else "white", edgecolors=col,
                   linewidths=0.9)
        ax.text(hi + 0.008, yi, st.fmt(r["diff"], sign=True) if abs(r["diff"]) >= 5e-4 else "0.000", va="center",
                ha="left", fontsize=6.6 if key else 6.0, color=ACCENT if key else INK2,
                fontweight="bold" if key else "normal")
        nums["a"][lab] = r
    sing = [r["diff"] for lab, r in rows if lab in ("monopolar target", "original scalp reference")]
    ki = [i for i, (lab, _) in enumerate(rows) if lab == KEY_ROW]
    if ki and len(sing) == 2:
        yk = y[ki[0]]
        add = sum(sing)
        ax.plot([add, add], [yk - 0.3, yk + 0.3], color=INK, lw=0.8, zorder=4)
        ax.text(add, yk + 0.36, "additive", ha="center", va="bottom", fontsize=5.6, color=INK)
        nums["a"]["additive_expectation"] = add
    ax.axvline(0, color=INK2, lw=0.5, zorder=0)
    ax.set_yticks(y, [lab for lab, _ in rows])
    for t, (lab, _) in zip(ax.get_yticklabels(), rows):
        t.set_color(ACCENT if lab == KEY_ROW else INK)
        t.set_fontweight("bold" if lab == KEY_ROW else "normal")
    st.bare_y(ax)
    ax.set_ylim(-0.6, len(rows) - 0.4)
    ax.set_xlabel("change in mean test r")
    xmax = max(r["ci"][1] for _, r in rows)
    ax.set_xlim(min(-0.03, min(r["ci"][0] for _, r in rows) - 0.01), xmax + 0.06)
    st.range_frame(ax, y=False)
    st.headline(ax, "Changing target and reference together inflates r",
                tag=f"one change at a time · single encoder · {n} patients · filled = CI excludes 0")
    axa = ax

    # ---------------- b: composite emulations
    ax = fig.add_subplot(gs[0, 1])
    ours_k = "recipe/ours (clean, r_concat, all patients)"
    rec = [("ours", ours_k, OURS),
           ("random 90 %", "rand_trials90/r_concat", BASE),
           ("chronological 90 %", "chrono90/r_concat", BASE),
           ("NeuroFlowNet-like", "recipe/NeuroFlowNet-like (random 90 % trials, target de-normalisation, flattened r)", BASE),
           ("monopolar + original ref.", "mono_orig/r_concat", ACCENT),
           ("CAST-like", "recipe/CAST-like (monopolar, orig ref, random segments, viable patients)", ACCENT)]
    rec = [r for r in rec if r[1] in summ]
    base_r = summ[ours_k]["enc"]
    yy = np.arange(len(rec))[::-1].astype(float)
    nums["b"] = {}
    for yi, (lab, k, col) in zip(yy, rec):
        v = summ[k]["enc"]
        ax.plot([0, v], [yi, yi], color=col if col != BASE else "#b3aea5", lw=2.6, solid_capstyle="butt", zorder=2)
        ax.text(v + 0.012, yi, f"{v:.2f}", va="center", fontsize=6.1, color=col if col != BASE else INK2,
                fontweight="bold" if col != BASE else "normal")
        ax.text(v + 0.075, yi, f"{v / base_r:.1f}×" if lab != "ours" else "", va="center", fontsize=5.6, color=INK3)
        nums["b"][lab] = {"r": v, "ratio": v / base_r}
    nk = "recipe/NeuroFlowNet metric on RANDOM predictions"
    if nk in summ:
        xn = summ[nk]["enc"]
        ax.plot([xn, xn], [yy[-1] - 0.5, yy[0] + 0.5], color=INK2, lw=0.5, ls=(0, (2, 1.5)), zorder=3)
        ax.text(xn + 0.008, yy[-1] - 0.5, "random predictions, flattened metric", fontsize=5.4, color=INK2,
                va="top", ha="left")
        nums["b"]["random_predictions_flattened"] = xn
    ax.set_yticks(yy, [r[0] for r in rec])
    for t, r in zip(ax.get_yticklabels(), rec):
        t.set_color(r[2] if r[2] != BASE else INK)
    st.bare_y(ax)
    ax.set_ylim(yy[-1] - 1.0, yy[0] + 0.6)
    ax.set_xlim(0, 0.62)
    ax.set_xticks([0, 0.2, 0.4, 0.6])
    ax.set_xlabel("mean test r, same model")
    st.range_frame(ax, y=False)
    st.headline(ax, "Literature-inspired protocols", tag="emulations, not reproductions")
    axb = ax

    # ---------------- c: per patient, our protocol -> monopolar target + original reference
    ax = fig.add_subplot(gs[1, 1])
    axc = ax
    if per:
        subs = sorted(per)
        a_ = np.array([per[s]["clean"]["enc"]["r_concat"] for s in subs])
        b_ = np.array([per[s]["mono_orig"]["enc"]["r_concat"] for s in subs])
        jump = b_ - a_
        top = set(np.argsort(-jump)[:4].tolist())
        for i in range(len(subs)):
            hot = i in top
            ax.plot([0, 1], [a_[i], b_[i]], color=ACCENT if hot else PATIENT, lw=0.9 if hot else 0.5, zorder=2 if hot else 1)
            ax.scatter([0, 1], [a_[i], b_[i]], s=5 if hot else 3, color=ACCENT if hot else PATIENT, linewidths=0,
                       zorder=3)
        lo_, hi_ = np.floor(min(a_.min(), b_.min()) / 0.2) * 0.2, np.ceil(max(a_.max(), b_.max()) / 0.2) * 0.2
        lab = sorted((b_[i], _short(subs[i])) for i in top)
        ys = _spread([v for v, _ in lab], 0.08 * (hi_ - lo_ + 0.04))
        for (v, s), yl in zip(lab, ys):
            ax.text(1.08, yl, s, fontsize=5.8, color=ACCENT, va="center")
        ax.set_xticks([0, 1], ["ours", "monopolar +\noriginal ref."])
        ax.tick_params(axis="x", length=0, pad=3)
        ax.spines["bottom"].set_visible(False)
        ax.set_xlim(-0.3, 1.4)
        ax.set_ylim(lo_ - 0.02, hi_ + 0.02)
        ax.set_yticks(np.round(np.arange(lo_, hi_ + 1e-9, 0.2), 1))
        ax.set_ylabel("test r")
        st.range_frame(ax, x=False)
        nums["c"] = {"top4": sorted(_short(subs[i]) for i in top), "jump_top4": [float(jump[i]) for i in sorted(top)]}
        st.headline(ax, "Carried by four patients", tag="lines = patients")
    st.letters(fig, [(axa, "a", -1.5), (axb, "b", -1.5), (axc, "c", -1.5)])
    st.save(fig, out, "fig6_protocol")
    dump_numbers(out, "fig6_protocol", nums)


# ============================================================================= figure 7
def _interaction(p_):
    return p_["mono|orig"] - p_["mono|car"] - p_["bip|orig"] + p_["bip|car"]


def _p17_load(R, name):
    out = {}
    for f in glob.glob(os.path.join(R, name, "readout_sub-P*.json")):
        d = json.load(open(f))
        out[d["subject"]] = d
    return out


def _p17_ex(D, m):
    return {s: r["explicit"][m]["mean"]["r"] for s, r in D.items() if m in r.get("explicit", {})}


def _ds_findings(runs):
    """The same five comparisons on ds004752 (primary level, strict; interaction: volts, lagged ridge)."""
    main_run, noenv = _runs(runs)
    S5 = load_strict(main_run)
    L = lambda name: load_dir(os.path.join(noenv, name + STRICT), "readout_")   # noqa: E731
    SF, F3, L3 = L("readout_fixed_strict"), L("readout_fixed_arch3_strict"), L("readout_learned_arch3_strict")
    out = {}

    def add(key, a, b):
        if a and b:
            _, (x, y) = paired(a, b)
            out[key] = paired_stats(x, y)
    add("h1", model_r(S5, "combo_nosess"), model_r(S5, "base"))
    add("h2", model_r(L3, "combo_nosess"), model_r(F3, "combo_nosess"))
    add("fixed1", model_r(S5, "combo_nosess"), model_r(SF, "combo_nosess"))
    add("half", model_r(S5, "combo_nosess", "0.1"), model_r(S5, "base", "0.2"))
    V = load_dir(os.path.join(runs, "review13"), "volts_")
    cm = {}
    if V:
        it = np.array([_interaction(v["pred"]) for v in V.values()])
        out["inter"] = paired_stats(it, np.zeros_like(it))
        out["inter"]["source"] = "review13 volts, lagged ridge"
        cm = {s: (abs(v["r_scalp_cm_vs_ieeg_cm"]), _interaction(v["pred"])) for s, v in V.items()}
    return out, cm


def fig7_external(runs, out):
    R = os.path.join(runs, P17)
    rp = os.path.join(R, "p17_report.json")
    if not os.path.exists(rp):
        raise FileNotFoundError(2, "missing", rp)
    rep = json.load(open(rp))
    setch_p = os.path.join(R, "p17_setch_report.json")
    setch = json.load(open(setch_p)) if os.path.exists(setch_p) else {}
    desc = {d["comparison"]: d for d in rep.get("descriptive", []) if d.get("level") == "primary"}
    ext = {"h1": rep["H1"], "h2": rep["H2"], "fixed1": desc.get("Final − Fixed-1"),
           "half": next((v for k, v in desc.items() if k.startswith("Final, 10 %")), None)}
    cm17 = rep.get("common_mode", {})
    if cm17:
        it = np.array([_interaction(v["pred"]) for v in cm17.values()])
        ext["inter"] = paired_stats(it, np.zeros_like(it))
    add = {"h1": setch.get("primary_setch|H1: Final − Base"), "h2": setch.get("primary_setch|H2: Learned-3 − Fixed-3")}
    ds, cm_ds = _ds_findings(runs)
    nums = {"ds004752": ds, "ScaleOLDNEW": {k: v for k, v in ext.items() if v}, "addendum_A": add}

    fig = plt.figure(figsize=(W2, 70 * MM))
    gs = fig.add_gridspec(1, 3, width_ratios=[0.52, 1.58, 0.82], wspace=0.3, left=0.07, right=0.985, top=0.85,
                          bottom=0.15)

    # ---------------- a: H1 per patient
    ax = fig.add_subplot(gs[0])
    F = _p17_load(os.path.join(R, "main"), "final_primary")
    b17, f17 = _p17_ex(F, "base"), _p17_ex(F, "combo_nosess")
    subs, (bb, ff) = paired(b17, f17)
    for b, f in zip(bb, ff):
        ax.plot([0, 1], [b, f], color="#a9cbc2", lw=0.5, zorder=1)
        ax.scatter([0, 1], [b, f], s=3, color="#a9cbc2", linewidths=0, zorder=2)
    if len(subs):
        ax.plot([0, 1], [bb.mean(), ff.mean()], color=EXT, lw=1.4, zorder=3)
        ax.scatter([0, 1], [bb.mean(), ff.mean()], s=16, color=[BASE, EXT], edgecolors="white", linewidths=0.6, zorder=4)
        ax.text(-0.1, bb.mean(), st.fmt(bb.mean()), ha="right", va="center", fontsize=6.0, color=BASE, fontweight="bold")
        ax.text(1.1, ff.mean(), st.fmt(ff.mean()), ha="left", va="center", fontsize=6.0, color=EXT, fontweight="bold")
    h = rep["H1"]
    ax.set_xticks([0, 1], ["ridge", "final"])
    ax.tick_params(axis="x", length=0, pad=3)
    ax.spines["bottom"].set_visible(False)
    ax.set_xlim(-0.55, 1.55)
    ax.set_ylabel("test r")
    st.range_frame(ax, x=False)
    st.headline(ax, "Every patient gains",
                tag=f"ScaleOLDNEW · H1 {st.fmt(h['mean_diff'], sign=True)}, {h['wins']}/{h['n']}")
    nums["a"] = {"n": len(subs), "mean_base": float(bb.mean()) if len(subs) else None,
                 "mean_final": float(ff.mean()) if len(subs) else None}
    axa = ax

    # ---------------- b: each development finding in both cohorts (labels inside the axis, above each row)
    ax = fig.add_subplot(gs[1])
    rows = [("h1", "final − ridge (H1)", True), ("h2", "learned − fixed, 3 banks (H2)", True),
            ("fixed1", "final − fixed bank", True), ("half", "half calibration − ridge", False),
            ("inter", "target × reference", False)]
    ypos, y = [], 0.0
    for i, (k, lab, rep_) in enumerate(rows):
        if i == 3:
            y -= 0.75                                     # gap between the two groups
        ypos.append(y)
        y -= 1.3
    ds_c, ex_c = INK3, EXT
    allx = [v["ci"][i] for d in (ds, ext) for v in d.values() if v for i in (0, 1)]
    x0 = min(-0.03, min(allx) - 0.01)
    # value columns on the right (x in axes units, y in data); headers double as the legend
    cols = [("ds004752", ds, ds_c, True, INK2), ("ScaleOLDNEW", ext, ex_c, True, EXT), ("subset", add, ex_c, False, EXT)]
    if not any(add.values()):
        cols = cols[:2]
    cx = NUDGE["fig7"].get("cols", [0.465, 0.625, 0.885])
    yax = ax.get_yaxis_transform()
    halo = dict(fc="white", ec="none", pad=0.6)
    for (k, lab, rep_), yi in zip(rows, ypos):
        ax.text(x0, yi + 0.24, lab, fontsize=6.2, color=INK if rep_ else INK2, va="bottom", ha="left", zorder=5,
                bbox=halo)
        for dy, src, col, filled in ((0.0, ds.get(k), ds_c, True), (-0.22, ext.get(k), ex_c, True),
                                     (-0.44, add.get(k), ex_c, False)):
            if not src:
                continue
            lo, hi = src["ci"]
            ax.plot([lo, hi], [yi + dy, yi + dy], color=col, lw=1.1 if filled else 0.7, solid_capstyle="butt", zorder=2)
            ax.scatter([src["mean_diff"]], [yi + dy], s=15 if filled else 10, marker="o" if filled else "D",
                       color=col if filled else "white", edgecolors=col, linewidths=0.8, zorder=3)
        for (cn, D, col, filled, tc), x_ in zip(cols, cx):
            v = (D.get(k) or {}).get("mean_diff")
            if v is not None:
                ax.text(x_ + 0.03, yi + 0.24, st.fmt(v, sign=True), transform=yax, fontsize=6.0, color=tc, va="bottom",
                        ha="left", fontweight="bold" if (rep_ and cn == "ScaleOLDNEW") else "normal")
    ax.axvline(0, color=INK2, lw=0.5, zorder=0)
    yh = ypos[0] + 0.95
    ax.text(x0, yh, "replicated", fontsize=6.4, color=EXT, fontweight="bold", va="bottom", ha="left", zorder=5,
            bbox=halo)
    ax.text(x0, ypos[3] + 0.95, "did not replicate", fontsize=6.4, color=INK2, fontweight="bold", va="bottom",
            ha="left", zorder=5, bbox=halo)
    for (cn, D, col, filled, tc), x_ in zip(cols, cx):
        ax.scatter([x_ + 0.01], [yh + 0.17], s=15 if filled else 10, marker="o" if filled else "D", transform=yax,
                   color=col if filled else "white", edgecolors=col, linewidths=0.8, zorder=4, clip_on=False)
        ax.text(x_ + 0.03, yh, cn, transform=yax, fontsize=5.8, color=tc, va="bottom", ha="left",
                fontweight="bold" if cn == "ScaleOLDNEW" else "normal")
    ax.set_yticks([])
    ax.spines["left"].set_visible(False)
    ax.set_ylim(ypos[-1] - 0.6, ypos[0] + 1.35)
    ax.set_xlim(x0, max(allx) + 0.03)
    ax.set_xlabel("difference in mean test r (95 % CI)")
    st.range_frame(ax, y=False)
    st.headline(ax, "Both pre-specified effects replicate at the same size",
                tag="primary level · strict · ScaleOLDNEW never seen during development")
    axb = ax

    # ---------------- c: common mode shared across modalities vs the target x reference interaction
    ax = fig.add_subplot(gs[2])
    hot_ds, hot17 = [], []
    for s, (x, it) in cm_ds.items():
        ax.scatter([x], [it], s=11, color=INK3, edgecolors="white", linewidths=0.4, zorder=2)
        if x > 0.6:
            hot_ds.append((x, it, _short(s)))
    for s, v in cm17.items():
        x, it = abs(v["r_scalp_cm_vs_ieeg_cm"]), _interaction(v["pred"])
        ax.scatter([x], [it], s=12, color=EXT, edgecolors="white", linewidths=0.4, zorder=3)
        if x > 0.6:
            hot17.append((x, it, _short(s)))
    for x, it, s in hot_ds + hot17:
        ax.scatter([x], [it], s=34, facecolors="none", edgecolors=ACCENT, linewidths=0.7, zorder=4)
    if hot_ds:
        ys = _spread([h[1] for h in hot_ds], 0.03)
        for (x, it, s), yl in zip(hot_ds, ys):
            ax.text(x - 0.07, yl, s, fontsize=5.6, color=ACCENT, ha="right", va="center")
    for x, it, s in hot17:
        ax.text(x - 0.07, it, s, fontsize=5.6, color=ACCENT, ha="right", va="center", fontweight="bold")
    st.ref_line(ax, 0)
    ax.set_xlim(-0.03, 1.03)
    ax.set_xticks([0, 0.5, 1.0])
    ax.set_xlabel("|r|, scalp vs SEEG common mode")
    ax.set_ylabel("target × reference interaction")
    ax.text(0.97, 0.24, "ds004752", transform=ax.transAxes, fontsize=5.8, color=INK2, ha="right")
    ax.text(0.97, 0.16, "ScaleOLDNEW", transform=ax.transAxes, fontsize=5.8, color=EXT, ha="right", fontweight="bold")
    st.range_frame(ax)
    st.headline(ax, "Where the interaction comes from", tag="per patient · volts · lagged ridge")
    nums["c"] = {"ds004752": {s: list(v) for s, v in cm_ds.items()},
                 "ScaleOLDNEW": {s: [abs(v["r_scalp_cm_vs_ieeg_cm"]), _interaction(v["pred"])] for s, v in cm17.items()}}
    axc = ax

    st.letters(fig, [(axa, "a", -1.0), (axb, "b", -4.0), (axc, "c", -1.0)])
    st.save(fig, out, "fig7_external")
    dump_numbers(out, "fig7_external", nums)


# ============================================================================= figure S1
def figS1_chanqc(runs, out):
    import csv
    p = os.path.join(runs, "chanqc20", "chanqc20_channels_v20_1.csv")
    if not os.path.exists(p):
        raise FileNotFoundError(2, "missing", p)
    rows = [r for r in csv.DictReader(open(p, encoding="utf-8")) if r["group"] != "no 10-05 name"]

    def num(r, k):
        try:
            return float(r[k])
        except (TypeError, ValueError):
            return np.nan
    groups = [("kept, listed in authors' file", lambda g: g == "kept, in .set", "#c3beb5", 5, 0.4),
              ("kept, not listed", lambda g: g == "kept, not in .set", OURS, 11, 0.9),
              ("not kept", lambda g: g.startswith("not kept"), ACCENT, 9, 0.9)]
    fig = plt.figure(figsize=(120 * MM, 80 * MM))
    ax = fig.add_axes([0.12, 0.15, 0.85, 0.72])
    nums = {}
    for lab, fn, col, s, a in groups:
        R_ = [r for r in rows if fn(r["group"])]
        x = np.array([num(r, "raw_sd_uV") for r in R_])
        y = np.clip(np.array([num(r, "r_unnamed_max") for r in R_]), 1e-3, 0.9995)
        ok = np.isfinite(x) & np.isfinite(y)
        ax.scatter(x[ok], y[ok], s=s, color=col, alpha=a, edgecolors="white" if s > 6 else "none", linewidths=0.4,
                   zorder=3 if s > 6 else 2)
        both = int(((x > 500) & (y >= 0.99)).sum())
        nums[lab] = {"n": int(ok.sum()), "raw_sd_gt500_and_r_ge099": both,
                     "max_raw_sd": float(np.nanmax(x)) if len(x) else None}
    ax.set_xscale("log")
    ax.set_yscale("logit")
    ax.set_ylim(6e-4, 0.9993)
    ax.set_yticks([0.001, 0.01, 0.1, 0.5, 0.9, 0.99, 0.999], ["0.001", "0.01", "0.1", "0.5", "0.9", "0.99", "0.999"])
    ax.minorticks_off()
    ax.set_xticks([10, 30, 100, 300, 1000, 3000], ["10", "30", "100", "300", "1000", "3000"])
    for v, axis in ((500, "x"), (0.99, "y")):
        st.ref_line(ax, v, axis=axis, color=INK2, lw=0.6)
    ax.text(500 / 1.06, 0.9975, "500 µV", fontsize=5.6, color=INK2, va="center", ha="right")
    ax.text(ax.get_xlim()[0] * 1.1 if ax.get_xlim()[0] > 0 else 6, 0.99 * 1.0, "|r| 0.99", fontsize=5.6, color=INK2,
            va="bottom", ha="left")
    ax.set_xlabel("raw standard deviation (µV)")
    ax.set_ylabel("max |r| with inputs without a 10-05 name")
    st.range_frame(ax)
    st.headline(ax, "Raw amplitude and similarity to unnamed inputs",
                tag="ScaleOLDNEW patients 01–07 · descriptive · both thresholds post hoc")
    # legend as coloured text in the empty lower right (high amplitude, low similarity)
    for i, (lab, fn, col, s, a) in enumerate(groups):
        ax.text(0.985, 0.30 - i * 0.075, f"{lab}  ({nums[lab]['n']})", transform=ax.transAxes, ha="right", va="center",
                color=col if col != "#c3beb5" else INK2, fontsize=6.0, zorder=5,
                bbox=dict(fc="white", ec="none", pad=0.8))
    st.save(fig, out, "figS1_chanqc")
    dump_numbers(out, "figS1_chanqc", nums)


FIGS = {"fig3": fig3_transfer, "fig4": fig4_reliab, "fig5": fig5_power, "fig6": fig6_protocol, "fig7": fig7_external,
        "figS1": figS1_chanqc}

## 3. Kiểm tra dữ liệu
Liệt kê mọi thư mục mà các hình cần, số file, và các khoá quan trọng. **Chụp hoặc copy toàn bộ output của ô này gửi lại**: nó cho biết hình nào sẽ đủ dữ liệu, nhất là `review13/volts_sub-*.json` của ds004752 (Hình 7).

In [ ]:
#@title 3. Kiểm tra dữ liệu (inventory)
import glob, json, os
R = RUNS_DIR
Q = '_qc_none'

def n_files(path, pat):
    return len(glob.glob(os.path.join(R, path, pat)))

def first(path, pat):
    f = sorted(glob.glob(os.path.join(R, path, pat)))
    return json.load(open(f[0])) if f else None

def row(fig, path, pat, extra=''):
    n = n_files(path, pat)
    print(f"{'OK   ' if n else 'THIẾU'} {fig:<7} {path + '/' + pat:<62} {n:>4} file  {extra}")

def frac_info(path):
    d = first(path, 'readout_sub-*.json')
    if not d:
        return ''
    fr = d.get('fracs', {})
    k = sorted(fr)
    mods = sorted(fr[k[0]]['models']) if k else []
    cv = any('cv_contacts' in v for f in fr.values() for v in f['models'].values())
    return f"fracs={k} models={mods}" + (" cv_contacts" if cv else "")

print('== ds004752 (development) ==')
for suf in ('', '_f10', '_f05'):
    p = f'shared_bip_car/readout_combo5_strict{Q}{suf}'
    row('2,3,4', p, 'readout_sub-*.json', frac_info(p))
row('2', 'shared_bip_car/readout_combo5', 'readout_sub-*.json', frac_info('shared_bip_car/readout_combo5'))
for name in ('readout_fixed_strict', 'readout_fixed_arch3_strict', 'readout_learned_arch3_strict'):
    p = f'shared_no_env/{name}{Q}'
    row('2,3,7', p, 'readout_sub-*.json', frac_info(p))
row('2', 'shared_no_env/readout_fixed', 'readout_sub-*.json')
row('2', 'nfn_baseline', 'nfn_sub-*.json')
row('3', 'shared_bip_car', 'enc_sub-*.pt')
d = first('shared_bip_car/readout_longevity', 'readout_sub-*.json')
row('4', 'shared_bip_car/readout_longevity', 'readout_sub-*.json',
    ('session_order ok' if d and 'session_order' in d else ''))
row('5', 'zeroshot_power', 'zeroshot_subjects.json')
row('6', 'protocol', 'protocol.json')
row('6', 'protocol', 'protocol_sub-*.json')
vs = sorted(os.path.basename(f)[6:-5] for f in glob.glob(os.path.join(R, 'review13', 'volts_*.json')))
row('7', 'review13', 'volts_*.json', ' '.join(vs))
lf = []
for run in sorted(glob.glob(os.path.join(R, '*'))):
    c = os.path.join(run, 'config.json')
    if os.path.exists(c):
        cfg = json.load(open(c))
        if cfg.get('target') == 'bip' and cfg.get('eeg_ref') == 'car' and 'calib_frac' in cfg:
            lf.append(f"{os.path.basename(run)} ({len(glob.glob(os.path.join(run, 'fold_sub-*.json')))} folds)")
print(f"{'OK   ' if lf else 'THIẾU'} 2       LeadFormer run (bip, car, calib_frac): {', '.join(lf) or '-'}")

print('\n== ScaleOLDNEW (external) ==')
row('7', 'p17', 'p17_report.json')
row('7', 'p17', 'p17_setch_report.json', '(addendum A, không bắt buộc)')
row('7', 'p17/main/final_primary', 'readout_sub-P*.json')
rp = os.path.join(R, 'p17', 'p17_report.json')
if os.path.exists(rp):
    rep = json.load(open(rp))
    print('      p17_report keys:', sorted(rep))
    print('      descriptive:', [(x.get('level'), x.get('comparison')) for x in rep.get('descriptive', [])][:12])
    print('      common_mode patients:', sorted(rep.get('common_mode', {})))
row('S1', 'chanqc20', 'chanqc20_channels_v20_1.csv')

## 4. Vẽ
Hình nào thiếu file đầu vào sẽ bị bỏ qua (in tên file thiếu); hình nào lỗi sẽ in traceback rồi vẽ tiếp các hình còn lại.
Muốn vẽ lại vài hình, điền `ONLY`, ví dụ `fig3 fig7`.

In [ ]:
#@title 4. Vẽ toàn bộ hình
ONLY = ''   #@param {type:"string"}
STYLE = 'own'   #@param ["own", "sp"]
only = f'--only {ONLY}' if ONLY.strip() else ''
!python -m s2i.paperfigs --runs "{RUNS_DIR}" --out "{FIG_DIR}" --style {STYLE} {only}

In [ ]:
#@title 5. Xem hình
from IPython.display import Image, display
NAMES = ['fig1_overview', 'fig2_main', 'fig3_transfer', 'fig4_reliability', 'fig5_band_power', 'fig6_protocol',
         'fig7_external', 'figS1_chanqc']
for name in NAMES:
    p = f'{FIG_DIR}/{name}.png'
    if os.path.exists(p):
        print(name)
        display(Image(p, width=1100))
    else:
        print(name, '-> chưa có')

In [ ]:
#@title 6. Gói PNG + numbers.json để gửi lại
import shutil, tempfile
from google.colab import files
tmp = tempfile.mkdtemp()
for f in glob.glob(f'{FIG_DIR}/*.png') + glob.glob(f'{FIG_DIR}/*_numbers.json'):
    shutil.copy(f, tmp)
z = shutil.make_archive('/content/s2i_figs', 'zip', tmp)
print(z, round(os.path.getsize(z) / 1e6, 1), 'MB')
files.download(z)

## Sau khi chạy
- Gửi lại **output ô 3** và file **`s2i_figs.zip`** (ô 6). Tôi sẽ đối chiếu từng `_numbers.json` với số trong manuscript, rồi chỉnh nhãn nếu khoảng giá trị thật làm chữ đè lên nhau.
- Vị trí nhãn tinh chỉnh tay nằm trong `NUDGE` ở đầu `paperfigs_b.py` (và `paperfigs.py` cho Hình 1–2).
- LaTeX: `\includegraphics[width=\textwidth]{figs/fig2_main.pdf}`. Không co giãn hình, để giữ cỡ chữ 6–8 pt.

**Số hình trong bản thảo v11** (tên file giữ nguyên):

| File | Bản thảo v11 |
|---|---|
| `fig1_overview` | Fig. 1 |
| `fig2_main` | Fig. 2 |
| `fig3_transfer` | Fig. 3 (cần cho đoạn mô tả bộ lọc đã học ở mục 3.2) |
| `fig6_protocol` | Fig. 4 |
| `fig7_external` | Fig. 5 |
| `fig4_reliability` | Supplementary Fig. S1 |
| `fig5_band_power` | Supplementary Fig. S2 |
| `figS1_chanqc` | Supplementary Fig. S3 |

`STYLE = 'sp'` dùng style SciencePlots (khung trục đầy đủ, tick hướng vào). Mặc định `'own'` là style đã thống nhất; chỉ thử `'sp'` nếu muốn so sánh, và khi đó cần `!pip -q install SciencePlots` trước.